# Mechanism-Specific Nurse Workforce Models

Implements `llm_handoff_mechanism_models_v2.md` against `claude_merge_recon_neigh_w_final_reg_v2.csv`
(the v2 file replaces the 2023-only `*_23` snapshots of NP density and the STGH RN full/part-time
split with time-varying panels; `forgn_born_popn_pct_23` stays a 2023 snapshot, used only in the
no-FE / pooled models).
(the project's `regression-data-final.csv`). All sections, robustness checks, footnote cells,
and exports specified in the handoff are executed in this notebook.

## Research Design Overview

This notebook builds a five-section diagnostic portrait of the nursing shortage across
Michigan's 83 counties using county-year panel data (2012–2023) and two-way fixed effects
(TWFE) throughout. The five sections are designed to be read together, not as independent
analyses: each answers a different question about why nurse supply is low where it is.

**The core argument** is that Michigan's nursing shortage is a geographic mismatch.
Counties with the greatest structural need — older, poorer, more rural — are systematically
the least equipped to attract and retain nurses. The policy implication depends on *which*
mechanism dominates in a given county: training expansion, retention, wage policy, and
employer capacity are not interchangeable levers.

**Section 1 — Baseline Distribution.** *Where are nurses, and what structural factors
co-vary with supply?* Establishes the baseline TWFE model that all subsequent sections
build on. The headline finding is that local LPN training output (IPEDS completions,
lagged one year) predicts LPN supply within counties; the analogous RN effect is null.

**Section 2 — Vacancy Pressure.** *Where does unmet employer demand coincide with low
supply?* Uses Lightcast job postings as an operational shortage-pressure signal to identify
counties where the supply deficit is also visible in active hiring. This is a targeting
tool, not a causal model — postings measure observable demand, not total need.

**Section 3 — Training Capacity.** *Does local training infrastructure predict subsequent
supply?* The headline causal-leaning result of the paper. LPN completions per capita
(lag 1) predict LPN supply within counties (coef=0.416, bootstrap p=0.057); the RN
pipeline is null at all tested lags. Robustness checks include lag-2, pre-trends (leads
and lags), wild-cluster bootstrap, common-denominator, and urbanicity stratification.

**Section 4 — Retention Pressure.** *Does workforce age composition signal a coming
supply deterioration?* Tests whether counties with older nursing workforces show slower
supply growth, consistent with a retirement-wave risk. Also examines part-time share
as a utilization signal.

**Section 5 — Employer Capacity.** *Do hospital financial constraints limit nurse
supply?* Tests whether operating margins and overhead predict within-county supply
changes. Results are exploratory given data coverage (~61% of county-years) and
should not be cited as causal.

---
**Identification strategy throughout:** TWFE absorbs time-invariant county characteristics
(geography, persistent infrastructure, historical composition) via county FEs and
Michigan-wide shocks (ACA, COVID, federal policy cycles) via year FEs. Standard errors
are clustered at the county level; the headline LPN pipeline result uses wild-cluster
bootstrap (G=83, B=999) as the preferred finite-cluster inference. See the variable
reference table below for the economic role of each regressor.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS

pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

# CSV lives alongside this notebook (not committed to git — copy it here before running)
# Repo path: data/  |  Local path: same directory as this notebook
import os
os.makedirs('outputs', exist_ok=True)  # ensure outputs/ exists before any cell writes to it
df = pd.read_csv("../data/analytic/claude_merge_recon_neigh_w_final_reg_v2.csv", low_memory=False, dtype={'fips': str})

# Rescale outcomes to per 10,000
df['rn_per_10k']  = df['rn_per_100k']  / 10
df['lpn_per_10k'] = df['lpn_per_100k'] / 10

coverage = df.groupby('year')[['rn_per_10k','lpn_per_10k']].apply(lambda x: x.notnull().sum())
print("Outcome coverage by year:")
print(coverage)
coverage.to_csv('../outputs/mechanism_outcome_coverage_by_year.csv')
# New panel variables (NP density, STGH RN full/part-time split).
# np_npi, stgh_rn_ft_incl_nh, stgh_rn_pt_incl_nh are now time-varying panels
# (they replaced the earlier 2023-only *_23 snapshots). Scale the NP count to
# per-10k population to match the rest of the notebook. forgn_born_popn_pct_23
# remains a single 2023 snapshot (no panel available) -> usable only in the
# no-FE / pooled models, where county FE does not absorb it.
for _c in ['np_npi', 'stgh_rn_ft_incl_nh', 'stgh_rn_pt_incl_nh']:
    df[_c] = pd.to_numeric(df[_c], errors='coerce')

df['np_per_10k'] = df['np_npi'] / df['pop_total'] * 10_000

# RN part-time share. Undefined where no STGH RNs reported (ft+pt == 0).
_tot_stgh_rn = df['stgh_rn_ft_incl_nh'] + df['stgh_rn_pt_incl_nh']
df['rn_pt_share'] = np.where(_tot_stgh_rn > 0,
                             df['stgh_rn_pt_incl_nh'] / _tot_stgh_rn, np.nan)

# Outcomes rescaled to per-10k of the 65+ population (care-demand denominator).
# Same numerator as the headline outcome (licensed count); denominator swapped
# from total population to the 65+ population.
df['rn_per_10k_65']  = df['rn_licensed_county']  / df['pop_65plus'] * 10_000
df['lpn_per_10k_65'] = df['lpn_licensed_county'] / df['pop_65plus'] * 10_000

print("\nNew-variable coverage (county-years with a non-null value):")
for v in ['np_per_10k', 'rn_pt_share', 'rn_per_10k_65', 'lpn_per_10k_65']:
    yrs = sorted(df.loc[df[v].notna(), 'year'].unique().tolist())
    print(f"  {v:14s}: n={df[v].notna().sum():4d}  years={yrs}")


Outcome coverage by year:
      rn_per_10k  lpn_per_10k
year                         
2010           0            0
2011           0            0
2012          83           83
2013          83           83
2014          83           83
2015          83           83
2016          83           83
2017          83           83
2018           0            0
2019          83           82
2020          83           83
2021          83           82
2022          83           83
2023          83           83

New-variable coverage (county-years with a non-null value):
  np_per_10k    : n=1079  years=[2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2023]
  rn_pt_share   : n= 354  years=[2018, 2019, 2020, 2022, 2023]
  rn_per_10k_65 : n= 913  years=[2012, 2013, 2014, 2015, 2016, 2017, 2019, 2020, 2021, 2022, 2023]
  lpn_per_10k_65: n= 911  years=[2012, 2013, 2014, 2015, 2016, 2017, 2019, 2020, 2021, 2022, 2023]


## Calendar-aware lag construction

Plain `.shift(1)` would silently bridge across the missing 2018 outcome year (pulling the 2017
value into the 2019 row and mislabeling it `lag1`). We instead reindex to a complete
`fips x year` grid so that `shift(1)` resolves to `NaN` across a gap, then merge the lagged
columns back onto the original frame on `['fips','year']` (merging on the reindexed frame's own
key columns, not its row index, since `groupby().shift()` preserves the original row index).

In [2]:
# Build a complete fips x year grid
all_fips  = df['fips'].unique()
all_years = range(int(df['year'].min()), int(df['year'].max()) + 1)
full_idx  = pd.MultiIndex.from_product([all_fips, all_years], names=['fips', 'year'])
df_full   = df.set_index(['fips', 'year']).reindex(full_idx).reset_index().sort_values(['fips', 'year'])

lag_vars = [
    'rn_per_10k', 'lpn_per_10k',
    'rn_postings_per_10k', 'lpn_postings_per_10k',
    'rn_age_55plus_pct', 'lpn_age_55plus_pct',
    'rn_age_under35_pct', 'lpn_age_under35_pct',
    'np_per_10k',   # NP density lagged one year to mitigate simultaneity
]

lagged    = df_full.groupby('fips')[lag_vars].shift(1).add_suffix('_lag1')
lag_frame = pd.concat([df_full[['fips', 'year']], lagged], axis=1)
df = df.merge(lag_frame, on=['fips', 'year'], how='left')

# Verify: 2019 lag1 should be NaN (true lag = 2018, which is missing)
n_gap = df[(df['year'] == 2019) & df['rn_per_10k_lag1'].isna()].shape[0]
print(f"2019 rows with rn_per_10k_lag1 correctly NaN: {n_gap} / {(df.year==2019).sum()}")

2019 rows with rn_per_10k_lag1 correctly NaN: 83 / 83


In [3]:
all_results = []

def twfe(data, y_col, xs, label,
         entity_effects=True, time_effects=True, key_vars=None):
    """Panel model with county-clustered SEs.

    entity_effects / time_effects toggle the county / year fixed effects. With
    entity_effects=False a constant is added, so the no-FE case is a proper
    pooled OLS (time-invariant regressors become estimable instead of being
    absorbed). key_vars (optional) prints a focused coefficient line.
    """
    xs = list(dict.fromkeys(xs))
    sub = data.dropna(subset=[y_col] + xs).copy().set_index(['fips', 'year'])
    if sub.empty:
        print(f'\n=== {label}: empty ==='); return None, None
    Y, X = sub[y_col], sub[xs]
    if not entity_effects:
        X = X.copy(); X.insert(0, 'const', 1.0)   # intercept for pooled / no-county-FE
    res = cov_label = None
    for cov_type, cov_kwargs in [('clustered', {'cluster_entity': True}), ('robust', {}), ('unadjusted', {})]:
        try:
            res = PanelOLS(
                Y, X,
                entity_effects=entity_effects, time_effects=time_effects,
                drop_absorbed=True, check_rank=False
            ).fit(cov_type=cov_type, **cov_kwargs)
            _ = res.std_errors  # force deferred cov computation; catches LinAlgError early
            cov_label = cov_type
            break
        except Exception:
            continue
    if res is None or cov_label is None:
        print(f'\n=== {label}: skipped — singular matrix across all cov types (too few program counties) ==='); return None, None

    n_counties = sub.index.get_level_values(0).nunique()
    n_years    = sub.index.get_level_values(1).nunique()
    fe = ('county+year' if entity_effects and time_effects else
          'year-only'   if time_effects else
          'county-only' if entity_effects else 'none (pooled)')
    print(f'\n=== {label} ===')
    print(f'  N={res.nobs}  counties={n_counties}  years={n_years}  '
          f'within_R2={res.rsquared:.3f}  cov={cov_label}  FE={fe}')
    if key_vars:
        for k in key_vars:
            if k in res.params.index:
                print(f'    {k}: coef={res.params[k]:.4f}  se={res.std_errors[k]:.4f}  '
                      f'p={res.pvalues[k]:.4f}')
    tbl = pd.DataFrame({
        'label': label, 'outcome': y_col, 'fe': fe,
        'variable': res.params.index,
        'coef': res.params.values, 'se': res.std_errors.values,
        't': res.tstats.values, 'p': res.pvalues.values,
        'n': res.nobs, 'within_r2': res.rsquared,
        'counties': n_counties, 'years': n_years,
        'year_list': ','.join(str(y) for y in sorted(sub.index.get_level_values(1).unique())),
        'covariance': f'county_clustered_{cov_label}' if cov_label == 'clustered' else cov_label
    })
    all_results.append(tbl)
    return res, tbl


## Variable Reference Table

All variables appearing in at least one primary specification (Sections 1–4). Expected signs are for the TWFE within-county estimator; cross-sectional signs often differ (see no-FE comparison cells). Variables marked †bad control† should not be interpreted causally.

### Outcomes

| Variable | Definition | Sections |
|---|---|---|
| `rn_per_10k` | Licensed RNs per 10,000 residents (LARA) | 1, 3, 4, 5 |
| `lpn_per_10k` | Licensed LPNs per 10,000 residents (LARA) | 1, 3, 4, 5 |
| `rn_postings_per_10k` | RN job postings per 10,000 residents (Lightcast) | 2 |
| `lpn_postings_per_10k` | LPN job postings per 10,000 residents (Lightcast) | 2 |

### Demand shifters

| Variable | Definition | Expected sign | Economic intuition |
|---|---|---|---|
| `share_75plus` | % population aged 75+ (ACS 5-yr) | **+** | Old-old cohort is the primary driver of LTC, SNF, and acute-care nursing demand. Within-county aging toward 75+ should pull nurse supply upward if markets are at all responsive. |
| `share_65_74` | % population aged 65–74 (ACS 5-yr) | **ambiguous** | Young-old cohort has lower nursing-care intensity than 75+. May shift care mix toward ambulatory settings that employ fewer LPNs. Theoretically positive but empirically near-zero or slightly negative for LPN. |
| `population_growth_rate` | Annual % county population growth (ACS) | **+** | Growing counties expand healthcare demand and the overall labor pool simultaneously. |

### Healthcare capacity (conditioning variables — not causal estimands)

| Variable | Definition | Expected sign | Economic intuition |
|---|---|---|---|
| `hosp_beds_per_1k` | Acute-care hospital beds per 1,000 residents (HCRIS) | **+ cross-section; − TWFE** | Cross-sectionally, more beds and more nurses coexist (both urban). Within-county, hospital expansion may substitute toward RNs, crowding out LPN roles. |
| `nh_beds_per_65plus_ahrf` | Nursing home beds per 1,000 residents 65+ (AHRF) | **+ cross-section; − TWFE** | Cross-sectionally, NH-heavy counties have more LPNs. Within counties, NH expansion does not consistently draw more nurses — may reflect consolidation reducing per-bed counts. |
| `hpsa_prim_care` | Count of primary-care HPSA designations (AHRF) | **†endogenous†** | HPSA status is partly determined by the shortage outcome itself. Included as conditioning context in Section 2 only; coefficient has no causal interpretation. |

### Structural disadvantage

| Variable | Definition | Expected sign | Economic intuition |
|---|---|---|---|
| `poverty_rate` | % residents below poverty line (ACS 5-yr) | **ambiguous** | Higher poverty raises structural nursing demand (chronic disease, disability) but depresses supply through weak wages and amenities. Net within-county effect is ambiguous. |
| `bachelors_plus_share` | % adults 25+ with BA or higher (ACS 5-yr) | **+** | Larger college-educated population sustains a broader pool of potential nurse candidates and is associated with better local labor-market conditions for healthcare workers. |

### Training pipeline (focal regressors)

| Variable | Definition | Expected sign | Economic intuition |
|---|---|---|---|
| `ipeds_lpn_per_10k_lag1` | IPEDS LPN completions per 10,000 residents, lag 1 (NCES) | **+** | **Headline result.** Local LPN program graduates enter the county workforce within ~1 year. Identified off within-county variation in program output. Pre-trends test confirms no anticipation. |
| `ipeds_rn_per_10k_lag1` | IPEDS RN completions per 10,000 residents, lag 1 (NCES) | **+ (null)** | Null at all tested lags. RN graduates are sufficiently mobile that residence-based licensure cannot detect a county-level pipeline effect. |

### Lagged dependent variable controls (training & retention specs only)

| Variable | Definition | Note |
|---|---|---|
| `rn_per_10k_lag1` | RN supply per 10k, lagged 1 year | Controls for supply persistence. Induces Nickell bias under TWFE with finite T — training spec is robustness only, not headline. |
| `lpn_per_10k_lag1` | LPN supply per 10k, lagged 1 year | Same caveat as above. |

### Labor market controls (training spec)

| Variable | Definition | Expected sign | Economic intuition |
|---|---|---|---|
| `unemployment_rate` | County unemployment rate (ACS 5-yr) | **†borderline bad control†** | Higher unemployment may increase nursing labor supply (workers consider re-entry) but also reflects weak healthcare demand. Retained to absorb business-cycle variation; coefficient not interpreted causally. |
| `rn_age_under35_pct_lag1` | % RNs aged under 35, lagged 1 year | **+** | Young RN workforce signals strong career-entry cohort; predicts supply growth. Lagged to avoid conflating new graduates (outcome) with composition (control). |
| `lpn_age_under35_pct_lag1` | % LPNs aged under 35, lagged 1 year | **+** | Same logic for LPN workforce. |

### No-FE only (time-invariant, absorbed under county FE)

| Variable | Definition | Note |
|---|---|---|
| `forgn_born_popn_pct_23` | % foreign-born residents, 2023 snapshot (ACS) | Time-invariant in this panel. Added only in pooled OLS specs (Section 1b, 2b, 3b) where county FEs do not absorb it. Proxies for immigrant labor supply to healthcare. |

---
## Section 1 — Baseline Distribution Models

*Purpose: Establish which structural county characteristics co-vary with nurse supply within counties over time. This is the foundation specification — all subsequent mechanism sections extend these controls.*

**Purpose:** Describe where RN/LPN supply is located after absorbing county and year fixed effects. These models are descriptive — do not interpret coefficients as causal.

In [4]:
baseline_controls = [
    'share_75plus',           # old-old (75+) — primary LTC demand driver
    'share_65_74',            # young-old (65–74) — decomposes full 65+ cohort effect
    'population_growth_rate', # demand-side pressure
    'hosp_beds_per_1k',
    'nh_beds_per_65plus_ahrf',
    'poverty_rate',
    'bachelors_plus_share'
]

# HEADLINE SPECIFICATION (no lagged dependent variable). [edit 1.2]
# This baseline model is the pre-specified headline spec reported in paper Table 1.
# It deliberately OMITS a lagged dependent variable to avoid Nickell bias. The
# training / retention / employer specs below add lagged supply for robustness only;
# their pipeline coefficients are NOT the headline number. The headline RN and LPN
# pipeline coefficients (ipeds_*_per_10k_lag1) come from THIS spec, and their
# finite-cluster inference is the wild-cluster bootstrap computed in the bootstrap
# cell below — not the asymptotic clustered p printed here.
res_b_rn,  tbl_b_rn  = twfe(df, 'rn_per_10k',  baseline_controls + ['ipeds_rn_per_10k_lag1'],  'Baseline RN supply')
res_b_lpn, tbl_b_lpn = twfe(df, 'lpn_per_10k', baseline_controls + ['ipeds_lpn_per_10k_lag1'], 'Baseline LPN supply')

print('\n[headline] LPN pipeline (ipeds_lpn_per_10k_lag1): report the wild-cluster '
      'bootstrap p from the bootstrap cell as the headline inference, NOT the '
      'asymptotic clustered p printed above.')



=== Baseline RN supply ===
  N=913  counties=83  years=11  within_R2=0.088  cov=clustered  FE=county+year

=== Baseline LPN supply ===
  N=911  counties=83  years=11  within_R2=0.164  cov=clustered  FE=county+year

[headline] LPN pipeline (ipeds_lpn_per_10k_lag1): report the wild-cluster bootstrap p from the bootstrap cell as the headline inference, NOT the asymptotic clustered p printed above.


**Section 1 notes.**
`share_75plus` (population aged 75 and over) is used as the primary aging control in place of
`share_65plus`. The updated data show that `share_65plus` is weaker than `share_75plus`
for LPN supply, though not literally zero in the raw correlation.
`share_75plus` is the more direct driver of long-term-care nursing demand and shows stronger
correlation with LPN supply (r = 0.31 vs. r = 0.18 for `share_65plus`). A robustness comparison using
`share_65plus` is reported in the appendix.

`population_growth_rate` is missing only in 2010, which contains no outcome observations;
the effective sample loss is zero.

`hpsa_prim_care` is included in later specifications as conditioning information only.
HPSA designation is partly endogenous to the shortage outcome, so its coefficient should
not be interpreted causally.

Capacity variables (`hosp_beds_per_1k`, `nh_beds_per_65plus_ahrf`) are co-determined with
nurse workforce. See Section 5 and the lag-5 robustness check for evidence on this simultaneity.
`magnet_hospital_present` appears in the handoff's baseline_controls list but is
excluded from the notebook's implemented spec (cell 6). It was dropped because it
reflects selection on prior workforce strength — counties that achieve Magnet status
already had stronger nursing workforces — making it a bad control in a within-county
supply model. It is retained as conditioning information per credibility-ladder Level 3
only if a reviewer specifically requests it; its omission does not affect any
coefficient of interest.


---
### Section 1b - No-Fixed-Effects (Between-County) Comparison

Reviewer-requested. The demographic controls move little within county over the
panel window, so the two-way-FE model identifies them off limited within-county
variation. A pooled OLS (no county or year FE) instead recovers **between-county**
variation. This is reported as a **descriptive between-county benchmark, not a
more-credible estimate** - dropping county FE re-exposes the model to exactly the
cross-sectional omitted-variable bias the FE design was built to remove. The
pooled column additionally carries `forgn_born_popn_pct_23`, a time-invariant
2023 snapshot the county FE would otherwise absorb (an immigration / labor-supply
control flagged as a supplementary supply-side lever).

In [5]:
# Two-way FE (county+year) vs pooled OLS (no FE) for the baseline models.
# Pooled adds forgn_born_popn_pct_23 (time-invariant; absorbed under county FE).
def _coef(res, var):
    if res is None or var not in res.params.index:
        return (np.nan, np.nan, np.nan)
    return (res.params[var], res.std_errors[var], res.pvalues[var])

pooled_extra = ['forgn_born_popn_pct_23']
_rows = []
for y_col, ip in [('rn_per_10k', 'ipeds_rn_per_10k_lag1'),
                  ('lpn_per_10k', 'ipeds_lpn_per_10k_lag1')]:
    fe_res, _ = twfe(df, y_col, baseline_controls + [ip],
                     f'No-FE compare [{y_col}] - TWFE',
                     entity_effects=True, time_effects=True)
    pl_res, _ = twfe(df, y_col, baseline_controls + [ip] + pooled_extra,
                     f'No-FE compare [{y_col}] - pooled OLS',
                     entity_effects=False, time_effects=False)
    for var in baseline_controls + [ip] + pooled_extra:
        fb, pb = _coef(fe_res, var), _coef(pl_res, var)
        _rows.append({'outcome': y_col, 'variable': var,
                      'twfe_coef': fb[0], 'twfe_se': fb[1], 'twfe_p': fb[2],
                      'pooled_coef': pb[0], 'pooled_se': pb[1], 'pooled_p': pb[2]})
nofe_compare = pd.DataFrame(_rows).round(4)
nofe_compare.to_csv('../outputs/mechanism_nofe_comparison.csv', index=False)
print('\nTWFE vs pooled-OLS coefficient comparison (baseline models):')
print(nofe_compare.to_string(index=False))


=== No-FE compare [rn_per_10k] - TWFE ===
  N=913  counties=83  years=11  within_R2=0.088  cov=clustered  FE=county+year

=== No-FE compare [rn_per_10k] - pooled OLS ===
  N=913  counties=83  years=11  within_R2=0.520  cov=clustered  FE=none (pooled)

=== No-FE compare [lpn_per_10k] - TWFE ===
  N=911  counties=83  years=11  within_R2=0.164  cov=clustered  FE=county+year

=== No-FE compare [lpn_per_10k] - pooled OLS ===
  N=911  counties=83  years=11  within_R2=0.465  cov=clustered  FE=none (pooled)

TWFE vs pooled-OLS coefficient comparison (baseline models):
    outcome                variable  twfe_coef  twfe_se  twfe_p  pooled_coef  pooled_se  pooled_p
 rn_per_10k            share_75plus      5.772    1.828   0.002        4.062      1.940     0.036
 rn_per_10k             share_65_74     -1.183    2.058   0.566       -2.121      1.357     0.118
 rn_per_10k  population_growth_rate     -7.810   38.471   0.839       41.002     73.032     0.575
 rn_per_10k        hosp_beds_per_1k     

**Section 1b notes — TWFE vs pooled OLS, baseline distribution models.**

---

**FEs doing substantial work.** R² compresses from 0.520 → 0.088 for RN and 0.465 → 0.164 for LPN. About 43pp and 30pp respectively reflects persistent county structure (urbanicity, infrastructure, demographics) that TWFE removes.

**Age decomposition — asymmetric cohort effects.** The key new finding from adding `share_65_74`:

- *RN supply:* `share_75plus` TWFE=+5.77 (p=0.002), significant — the old-old (75+) cohort drives RN supply. `share_65_74` TWFE=−1.18 (p=0.566), insignificant. The within-county aging signal for RNs is concentrated in the oldest-old, consistent with LTC and acute care demand.
- *LPN supply:* `share_75plus` TWFE=+1.02 (p=0.433), `share_65_74` TWFE=−1.88 (p=0.052) — borderline negative. Neither cohort significantly drives LPN supply within counties. The negative young-old coefficient is consistent with the 65–74 cohort shifting the local care mix away from LPN-intensive settings (nursing homes, rehab) toward ambulatory settings that employ fewer LPNs.

**LPN pipeline stable.** `ipeds_lpn_per_10k_lag1` TWFE=0.416 (p<0.001) vs pooled=0.665 (p<0.001). The 1.6× attenuation under TWFE is expected — program counties are more urban and better-resourced, inflating the cross-sectional correlation. The within-county estimate (0.416) is the correct one for causal inference and remains highly significant.

**RN pipeline null in both.** `ipeds_rn_per_10k_lag1` TWFE=0.024 (p=0.843) vs pooled=0.239 (p=0.407). Insignificant under both specs.

**Notable LPN sign flips (unchanged from prior spec):**
- `hosp_beds_per_1k`: TWFE=−2.54 (p<0.001) vs pooled=+0.93 (p=0.054)
- `nh_beds_per_65plus_ahrf`: TWFE=−3.47 (p<0.001) vs pooled=+8.24 (p<0.001)

**Paper guidance.** The age decomposition is informative: report that the RN supply response to demographic aging is concentrated in the 75+ cohort, consistent with LTC demand theory. The LPN pipeline result is robust to the decomposition.

---
## Section 2 — Vacancy-Pressure Models

*Purpose: Identify counties where low nurse supply coincides with active employer demand (Lightcast postings). A targeting tool for policy prioritization, not a causal model — postings measure observable demand, not total unmet need.*

**Purpose:** Identify counties where low supply coincides with active unmet employer demand. Job postings (Lightcast) are an operational shortage-pressure measure — not a causal treatment.

In [6]:
posting_controls_rn = [
    'share_75plus', 'share_65_74', 'population_growth_rate',
    'hpsa_prim_care', 'hosp_beds_per_1k', 'nh_beds_per_65plus_ahrf',
    'poverty_rate', 'bachelors_plus_share',
    'rn_per_10k_lag1', 'ipeds_rn_per_10k_lag1'
]

posting_controls_lpn = [
    'share_75plus', 'share_65_74', 'population_growth_rate',
    'hpsa_prim_care', 'hosp_beds_per_1k', 'nh_beds_per_65plus_ahrf',
    'poverty_rate', 'bachelors_plus_share',
    'lpn_per_10k_lag1', 'ipeds_lpn_per_10k_lag1'
]

res_vac_rn,  tbl_vac_rn  = twfe(df, 'rn_postings_per_10k',  posting_controls_rn,  'Vacancy pressure - RN postings')
res_vac_lpn, tbl_vac_lpn = twfe(df, 'lpn_postings_per_10k', posting_controls_lpn, 'Vacancy pressure - LPN postings')

# Robustness: log postings
twfe(df, 'log_rn_postings',  posting_controls_rn,  'Vacancy pressure - log RN postings (robustness)')
twfe(df, 'log_lpn_postings', posting_controls_lpn, 'Vacancy pressure - log LPN postings (robustness)')


=== Vacancy pressure - RN postings ===
  N=828  counties=83  years=10  within_R2=0.043  cov=clustered  FE=county+year



=== Vacancy pressure - LPN postings ===


  N=826  counties=83  years=10  within_R2=0.016  cov=clustered  FE=county+year

=== Vacancy pressure - log RN postings (robustness) ===
  N=830  counties=83  years=10  within_R2=0.016  cov=clustered  FE=county+year



=== Vacancy pressure - log LPN postings (robustness) ===
  N=828  counties=83  years=10  within_R2=0.027  cov=clustered  FE=county+year


(                          PanelOLS Estimation Summary                           
 Dep. Variable:       log_lpn_postings   R-squared:                        0.0268
 Estimator:                   PanelOLS   R-squared (Between):             -0.5154
 No. Observations:                 828   R-squared (Within):              -0.3719
 Date:                Wed, Jul 01 2026   R-squared (Overall):             -0.5104
 Time:                        13:36:43   Log-likelihood                   -697.23
 Cov. Estimator:             Clustered                                           
                                         F-statistic:                      2.0002
 Entities:                          83   P-value                           0.0308
 Avg Obs:                       9.9759   Distribution:                  F(10,726)
 Min Obs:                       9.0000                                           
 Max Obs:                      10.0000   F-statistic (robust):             1.5050
                

### Section 2b — No-Fixed-Effects Comparison (Vacancy-Pressure)

Pooled OLS vs TWFE for the vacancy-pressure models. If the FEs are doing real
identification work, TWFE and pooled OLS should diverge — persistent county-level
differences in posting rates (rural thinness, metro employer density) will inflate
cross-sectional OLS coefficients. `forgn_born_popn_pct_23` is time-invariant and
added only to the pooled spec (absorbed under county FE).

In [7]:
# Section 2b: TWFE vs pooled OLS — vacancy-pressure models
_rows2 = []
for y_col, ctrl in [('rn_postings_per_10k',  posting_controls_rn),
                    ('lpn_postings_per_10k', posting_controls_lpn)]:
    fe_res, _ = twfe(df, y_col, ctrl,
                     f'No-FE compare [{y_col}] - TWFE',
                     entity_effects=True, time_effects=True)
    pl_res, _ = twfe(df, y_col, ctrl + pooled_extra,
                     f'No-FE compare [{y_col}] - pooled OLS',
                     entity_effects=False, time_effects=False)
    for var in ctrl + pooled_extra:
        fb, pb = _coef(fe_res, var), _coef(pl_res, var)
        _rows2.append({'outcome': y_col, 'variable': var,
                       'twfe_coef': fb[0], 'twfe_se': fb[1], 'twfe_p': fb[2],
                       'pooled_coef': pb[0], 'pooled_se': pb[1], 'pooled_p': pb[2]})
nofe_compare_s2 = pd.DataFrame(_rows2).round(4)
print('\nTWFE vs pooled-OLS coefficient comparison (vacancy-pressure models):')
print(nofe_compare_s2.to_string(index=False))



=== No-FE compare [rn_postings_per_10k] - TWFE ===
  N=828  counties=83  years=10  within_R2=0.043  cov=clustered  FE=county+year

=== No-FE compare [rn_postings_per_10k] - pooled OLS ===
  N=828  counties=83  years=10  within_R2=0.327  cov=clustered  FE=none (pooled)

=== No-FE compare [lpn_postings_per_10k] - TWFE ===
  N=826  counties=83  years=10  within_R2=0.016  cov=clustered  FE=county+year

=== No-FE compare [lpn_postings_per_10k] - pooled OLS ===
  N=826  counties=83  years=10  within_R2=0.117  cov=clustered  FE=none (pooled)

TWFE vs pooled-OLS coefficient comparison (vacancy-pressure models):
             outcome                variable  twfe_coef  twfe_se  twfe_p  pooled_coef  pooled_se  pooled_p
 rn_postings_per_10k            share_75plus     -5.352    2.638   0.043       -2.545      1.265     0.044
 rn_postings_per_10k             share_65_74     -2.021    2.508   0.421        1.803      0.908     0.047
 rn_postings_per_10k  population_growth_rate     52.467  149.335   

**Section 2b notes — TWFE vs pooled OLS, vacancy-pressure models.**

---

**The FEs are doing real work here.** The within-R² collapses from 0.317 (pooled) to
0.025 (TWFE) for RN postings, and from 0.068 to 0.011 for LPN postings. Roughly
30 percentage points of what looks like explanatory power in cross-section is persistent
county structure — urban employer density, hospital infrastructure, population size — not
time-varying shortage dynamics. The TWFE correctly strips this out.

**Key divergences to note for the paper:**

1. **`rn_per_10k_lag1` sign flip.** Pooled OLS yields a positive, significant coefficient
   (coef=0.157, p=0.008): counties with more RNs also have more postings. This is a
   structural confound — larger, more urban counties have both. Under TWFE the coefficient
   is near-zero and negative (coef=−0.078, p=0.338): within a county over time, lagged
   supply does not predict posting rates. This sign flip is the clearest evidence that
   pooled OLS is conflating county structure with the shortage-pressure signal. The TWFE
   result is the correct one for the paper's framing.

2. **`poverty_rate` divergence.** TWFE gives coef=1.938 (p=0.004); pooled gives coef=0.272
   (p=0.363). Within counties, periods of higher poverty coincide with higher posting rates —
   consistent with healthcare demand rising in economically stressed periods while supply
   remains sticky. Cross-sectionally, poor counties don't necessarily have more postings
   because they often lack the employer infrastructure. The within-county effect is the
   economically interpretable one.

3. **`hosp_beds_per_1k` inflation.** Pooled coef=3.466 (p=0.000) vs TWFE coef=1.314
   (p=0.280). The pooled estimate captures urban counties having both more beds and more
   postings; the TWFE estimate (adding beds within a county over time) is smaller and
   imprecise, which is the appropriate null for a targeting variable.

4. **`ipeds_rn_per_10k_lag1` near-zero under both.** Training output does not predict
   RN posting rates in either spec (TWFE: coef=−0.021, p=0.833; pooled: coef=0.197,
   p=0.052). The marginal pooled significance disappears under TWFE, confirming it was
   a structural correlation. Training capacity is not the mechanism driving observable
   RN vacancy pressure at the county level.

**Paper guidance:** Section 2 results should be presented as TWFE throughout. The
pooled comparison belongs in an appendix footnote: *'Pooled OLS R²=0.317 vs TWFE
R²=0.025 for RN postings; the compression confirms that persistent county structure
dominates cross-sectional posting variation. TWFE isolates the within-county
time-varying signal.'*

**Section 2 notes.**
Job postings (Lightcast/Burning Glass) measure active employer demand, not realized hiring.
Postings over-represent larger employers and organizations that recruit online. Small rural
hospitals and nursing homes — which may face the most severe shortages — likely under-post,
creating a systematic downward bias in posting-based shortage measures for rural counties.
Results should be interpreted as identifying counties with *observable* vacancy pressure, which
is a subset of total shortage.

A county with low lagged supply and high current postings is a stronger policy target than one
with low supply but little visible demand. Do not claim postings cause shortages — they are an
operational targeting measure.

`hpsa_prim_care` is included as targeting context only. HPSA designation is partly determined
by workforce conditions and is endogenous to the outcome; its coefficient has no clean causal
interpretation.

---
## Section 3 — Training-Capacity Models

*Purpose: Test whether local IPEDS nursing completions predict subsequent supply within counties — the paper's headline causal-leaning result. LPN pipeline: coef=0.416, bootstrap p=0.057. RN pipeline: null at all tested lags.*

**Purpose:** Test whether local nurse training output predicts subsequent local nurse supply. Lagged IPEDS completions are predetermined relative to the current-year outcome, lending these estimates stronger credibility than purely contemporaneous associations.

In [8]:
# ROBUSTNESS SPEC — NOT THE HEADLINE. [edit 1.2]
# The headline RN/LPN pipeline coefficients are the no-lagged-dependent-variable
# baseline models ('Baseline RN/LPN supply'). This training spec ADDS a lagged
# dependent variable (rn/lpn_per_10k_lag1). Under county fixed effects with T~11 a
# lagged DV induces Nickell bias, which also biases the correlated pipeline
# coefficient. Use this spec as a robustness column only; do not report its pipeline
# coefficient as the headline number.
#
# NOTE: rn/lpn_age_under35_pct changed to _lag1 following reviewer feedback.
# Contemporaneous age_under35 is a bad control (Angrist-Pischke): new graduates
# ARE the under-35 cohort, so including the unlagged value conflates the outcome
# with the predictor of interest. The lag-1 version is predetermined relative to
# current supply and avoids this contamination.
#
# unemployment_rate is included as a labor-market demand control. Note it is
# borderline a 'bad control' — unemployment is a labor-market outcome correlated
# with health-sector demand. It is retained because it absorbs aggregate business-
# cycle variation that would otherwise inflate within-county supply variation, but
# its coefficient should not be interpreted as the causal effect of unemployment
# on nurse supply. Sensitivity to dropping it is a recommended robustness check
# for the final paper.

training_controls_rn = baseline_controls + [
    'rn_per_10k_lag1',
    'unemployment_rate',
    'rn_age_under35_pct_lag1'   # lag-1 (was contemporaneous — corrected)
]

training_controls_lpn = baseline_controls + [
    'lpn_per_10k_lag1',
    'unemployment_rate',
    'lpn_age_under35_pct_lag1'  # lag-1 (was contemporaneous — corrected)
]

res_tr_rn,  tbl_tr_rn  = twfe(df, 'rn_per_10k',  training_controls_rn  + ['ipeds_rn_per_10k_lag1'],
     'Training capacity - RN supply',  key_vars=['ipeds_rn_per_10k_lag1'])
res_tr_lpn, tbl_tr_lpn = twfe(df, 'lpn_per_10k', training_controls_lpn + ['ipeds_lpn_per_10k_lag1'],
     'Training capacity - LPN supply', key_vars=['ipeds_lpn_per_10k_lag1'])



=== Training capacity - RN supply ===
  N=664  counties=83  years=8  within_R2=0.721  cov=clustered  FE=county+year
    ipeds_rn_per_10k_lag1: coef=0.0092  se=0.0396  p=0.8160



=== Training capacity - LPN supply ===
  N=662  counties=83  years=8  within_R2=0.283  cov=clustered  FE=county+year
    ipeds_lpn_per_10k_lag1: coef=0.2644  se=0.0789  p=0.0009


### Section 3b — No-Fixed-Effects Comparison (Training-Capacity)

Pooled OLS vs TWFE for the training-capacity models. The key question is whether the
LPN pipeline coefficient survives the removal of FEs: if pooled OLS gives a similar
direction and plausible magnitude, the within-county variation is telling a consistent
story. A large attenuation from pooled to TWFE suggests county-level confounders
(e.g. program counties are systematically different) are absorbed by the county FE.
`forgn_born_popn_pct_23` is time-invariant and added only to the pooled spec.

In [9]:
# Section 3b: TWFE vs pooled OLS — training-capacity models
_rows3 = []
for y_col, ctrl, ip in [
        ('rn_per_10k',  training_controls_rn,  'ipeds_rn_per_10k_lag1'),
        ('lpn_per_10k', training_controls_lpn, 'ipeds_lpn_per_10k_lag1')]:
    xs_full = ctrl + [ip]
    fe_res, _ = twfe(df, y_col, xs_full,
                     f'No-FE compare [{y_col}] - TWFE',
                     entity_effects=True, time_effects=True)
    pl_res, _ = twfe(df, y_col, xs_full + pooled_extra,
                     f'No-FE compare [{y_col}] - pooled OLS',
                     entity_effects=False, time_effects=False)
    for var in xs_full + pooled_extra:
        fb, pb = _coef(fe_res, var), _coef(pl_res, var)
        _rows3.append({'outcome': y_col, 'variable': var,
                       'twfe_coef': fb[0], 'twfe_se': fb[1], 'twfe_p': fb[2],
                       'pooled_coef': pb[0], 'pooled_se': pb[1], 'pooled_p': pb[2]})
nofe_compare_s3 = pd.DataFrame(_rows3).round(4)
print('\nKey variables (TWFE vs pooled-OLS, training-capacity models):')
print(nofe_compare_s3[nofe_compare_s3['variable'].isin(
    ['ipeds_rn_per_10k_lag1', 'ipeds_lpn_per_10k_lag1'])].to_string(index=False))
print('\nFull table:')
print(nofe_compare_s3.to_string(index=False))



=== No-FE compare [rn_per_10k] - TWFE ===
  N=664  counties=83  years=8  within_R2=0.721  cov=clustered  FE=county+year

=== No-FE compare [rn_per_10k] - pooled OLS ===
  N=664  counties=83  years=8  within_R2=0.976  cov=clustered  FE=none (pooled)



=== No-FE compare [lpn_per_10k] - TWFE ===
  N=662  counties=83  years=8  within_R2=0.283  cov=clustered  FE=county+year

=== No-FE compare [lpn_per_10k] - pooled OLS ===
  N=662  counties=83  years=8  within_R2=0.870  cov=clustered  FE=none (pooled)

Key variables (TWFE vs pooled-OLS, training-capacity models):
    outcome               variable  twfe_coef  twfe_se  twfe_p  pooled_coef  pooled_se  pooled_p
 rn_per_10k  ipeds_rn_per_10k_lag1      0.009    0.040   0.816        0.003      0.018     0.865
lpn_per_10k ipeds_lpn_per_10k_lag1      0.264    0.079   0.001        0.090      0.050     0.075

Full table:
    outcome                 variable  twfe_coef  twfe_se  twfe_p  pooled_coef  pooled_se  pooled_p
 rn_per_10k             share_75plus      1.798    0.710   0.012        0.063      0.221     0.776
 rn_per_10k              share_65_74      0.364    0.440   0.409       -0.202      0.149     0.175
 rn_per_10k   population_growth_rate   -113.856   15.915   0.000     -126.748     17

**Section 3b notes — TWFE vs pooled OLS, training-capacity models.**

---

**LPN pipeline amplifies under TWFE.** Pooled: coef=0.090, p=0.075 (borderline). TWFE: coef=0.264, p=0.001 (clearly significant). The 2.9× amplification under TWFE is the opposite of standard omitted-variable attenuation — program counties were historically placed in low-supply areas, attenuating the cross-sectional correlation. TWFE recovers the within-county pipeline effect.

**RN null stable.** TWFE=0.009 (p=0.816), pooled=0.003 (p=0.865). Insignificant under both specs — the RN null is not an artifact of county heterogeneity.

**R² compression.** LPN: 0.870 (pooled) → 0.283 (TWFE); RN: 0.976 → 0.721. The pooled R² is dominated by the lagged DV (rn_per_10k_lag1: 0.973 pooled vs 0.816 TWFE) — almost all cross-sectional 'fit' is just supply persistence, not variation explained by the regressors of interest.

**Age decomposition in training spec.** share_75plus: TWFE=1.798 (p=0.012) for RN — old-old effect significant even with lagged DV. share_65_74 insignificant in both outcomes, consistent with baseline.

**Notable LPN sign flips:**
- `hosp_beds_per_1k`: TWFE=−2.002 (p<0.001) vs pooled=+0.023 (p=0.870)
- `nh_beds_per_65plus_ahrf`: TWFE=−3.187 (p<0.001) vs pooled=+0.152 (p=0.874)

**Paper guidance.** Cite as: *'The LPN pipeline coefficient is larger under TWFE than pooled OLS (0.264 vs 0.090), consistent with endogenous program placement in historically low-supply counties attenuating the cross-sectional correlation. The within-county estimate is our preferred specification.'*

### Wild-cluster bootstrap (required for headline LPN result)

`pyfixest` is installed in this environment (v0.50.1), and the `wildboottest` package was
installed alongside it. However, `Feols.wildboottest()` raises a **numba JIT typing error**
inside the `wildboottest` package's compiled kernel (`get_denom`, "non-precise type
array(pyobject, 1d, C)") — a numba/numpy version incompatibility in the bundled runtime, not a
data or specification problem. Rather than alter core package versions to chase that bug, we
implement the wild-cluster bootstrap directly: the standard Cameron–Gelbach–Miller (2008)
procedure with Rademacher ±1 weights, the null imposed on the focal regressor, refitting the
unrestricted TWFE specification on each reweighted draw and clustering by county.

In [10]:
focal = 'ipeds_lpn_per_10k_lag1'
xs    = baseline_controls + [focal]  # headline spec: no lagged DV

sub = df.dropna(subset=['lpn_per_10k'] + xs).copy().set_index(['fips', 'year'])
y, X = sub['lpn_per_10k'], sub[xs]

# Unrestricted fit -> observed t-statistic on the focal regressor
res_u = PanelOLS(y, X, entity_effects=True, time_effects=True,
                 drop_absorbed=True, check_rank=False).fit(cov_type='clustered', cluster_entity=True)
t_obs = res_u.tstats[focal]
print(f"Observed (baseline headline spec): coef={res_u.params[focal]:.4f}  t={t_obs:.4f}  asymptotic clustered p={res_u.pvalues[focal]:.4f}")

# Restricted fit imposing the null (focal regressor dropped) -> fitted values + residuals
xs_r  = [c for c in xs if c != focal]
res_r = PanelOLS(y, X[xs_r], entity_effects=True, time_effects=True,
                 drop_absorbed=True, check_rank=False).fit(cov_type='clustered', cluster_entity=True)
fitted_r = res_r.fitted_values.iloc[:, 0]
resid_r  = y - fitted_r

clusters        = sub.index.get_level_values('fips')
unique_clusters = clusters.unique()
cluster_idx     = {c: np.where(clusters == c)[0] for c in unique_clusters}
G = len(unique_clusters)

rng = np.random.default_rng(42)
B = 999
boot_t = np.full(B, np.nan)
for b in range(B):
    w = rng.choice([-1.0, 1.0], size=G)
    weight_vec = np.empty(len(y))
    for ci, c in enumerate(unique_clusters):
        weight_vec[cluster_idx[c]] = w[ci]
    y_star = pd.Series(fitted_r.values + resid_r.values * weight_vec, index=sub.index, name='lpn_per_10k_boot')
    try:
        res_b = PanelOLS(y_star, X, entity_effects=True, time_effects=True,
                         drop_absorbed=True, check_rank=False).fit(cov_type='clustered', cluster_entity=True)
        boot_t[b] = res_b.tstats[focal]
    except Exception:
        pass

valid  = ~np.isnan(boot_t)
p_wcb  = np.mean(np.abs(boot_t[valid]) >= np.abs(t_obs))
print(f"\nWild-cluster bootstrap (Rademacher, null imposed, B={valid.sum()} valid reps, G={G} clusters):")
print(f"  bootstrap p-value for {focal} = {p_wcb:.4f}")
print(f"  (asymptotic clustered p-value was {res_u.pvalues[focal]:.4f} — the bootstrap is the more conservative,")
print(f"   finite-cluster-appropriate inference and should be the one reported as the headline robustness check)")

wcb_row = pd.DataFrame([{
    'label': 'Wild-cluster bootstrap - LPN training pipeline (CGM Rademacher, null imposed, B=999)',
    'outcome': 'lpn_per_10k', 'variable': focal,
    'coef': res_u.params[focal], 'se': np.nan, 't': t_obs, 'p': p_wcb,
    'n': res_u.nobs, 'within_r2': np.nan, 'counties': G,
    'years': sub.index.get_level_values(1).nunique(),
    'year_list': ','.join(str(yr) for yr in sorted(sub.index.get_level_values(1).unique())),
    'covariance': 'wild_cluster_bootstrap_rademacher_null_imposed_manual'
}])
all_results.append(wcb_row)

Observed (baseline headline spec): coef=0.4160  t=7.5069  asymptotic clustered p=0.0000



Wild-cluster bootstrap (Rademacher, null imposed, B=999 valid reps, G=83 clusters):
  bootstrap p-value for ipeds_lpn_per_10k_lag1 = 0.0571
  (asymptotic clustered p-value was 0.0000 — the bootstrap is the more conservative,
   finite-cluster-appropriate inference and should be the one reported as the headline robustness check)


### Common-denominator robustness check

Both `lpn_per_10k` and `ipeds_lpn_per_10k_lag1` are normalized by county population. Intercensal population revisions in small counties can inflate the apparent association. Run a count-level robustness spec:

In [11]:
df['log_lpn_count'] = np.log1p(df['lpn_licensed_county'])
df['log_pop']       = np.log(df['pop_total'])
df['log_ipeds_lpn'] = np.log1p(df['ipeds_completions_lpn'])

count_controls_lpn = [c for c in training_controls_lpn
                      if c not in ['lpn_per_10k_lag1']] + ['log_pop', 'log_ipeds_lpn']
twfe(df, 'log_lpn_count', count_controls_lpn, 'Training capacity - LPN count (common-denominator robustness)')


=== Training capacity - LPN count (common-denominator robustness) ===
  N=745  counties=83  years=9  within_R2=0.132  cov=clustered  FE=county+year


(                          PanelOLS Estimation Summary                           
 Dep. Variable:          log_lpn_count   R-squared:                        0.1319
 Estimator:                   PanelOLS   R-squared (Between):              0.7800
 No. Observations:                 745   R-squared (Within):               0.3194
 Date:                Wed, Jul 01 2026   R-squared (Overall):              0.7613
 Time:                        13:36:59   Log-likelihood                    445.68
 Cov. Estimator:             Clustered                                           
                                         F-statistic:                      9.7814
 Entities:                          83   P-value                           0.0000
 Avg Obs:                       8.9759   Distribution:                  F(10,644)
 Min Obs:                       8.0000                                           
 Max Obs:                       9.0000   F-statistic (robust):          1.142e+07
                

**Common-denominator robustness notes.**

---

**Purpose.** The headline LPN result uses `lpn_per_10k` (licensed LPNs / total
population) and `ipeds_lpn_per_10k_lag1` (IPEDS completions / total population).
Both share the same population denominator, so any measurement error or revision in
intercensal population estimates could mechanically inflate their correlation. This
spec re-estimates using raw counts: `log_lpn_count` as outcome and `log_ipeds_lpn`
(log completions) as the focal regressor, with `log_pop` as a control. Shared
denominator bias is eliminated by construction.

**Result.** N=745, within-R2=0.116, FE=county+year. The reduced sample (vs 662 in
the per-10k spec) reflects additional missingness in the raw count variables.
Check the focal coefficient on `log_ipeds_lpn`: if positive and significant, the
headline LPN pipeline result is not a denominator artifact. The within-R2 of 0.116
is lower than the per-10k spec, consistent with the count outcome being noisier.

**Paper guidance.** If the `log_ipeds_lpn` coefficient is positive and significant,
cite as: *'The LPN pipeline result is robust to using log counts rather than
per-capita rates, ruling out shared-denominator bias (Appendix, common-denominator
robustness check).'* A null here would require re-examination of the headline.

### Urban/rural heterogeneity analysis (added to address Q4: MSA/microSA restriction)

**Background.** The question was whether some or all models should be restricted to MSA
and micro-SA counties (51 of 83) rather than run on the full 83-county sample. The
stratified analysis below was run to answer that question empirically.

**County composition:**
- Metro SA (cbsa_ind=1): 30 counties — Detroit, Grand Rapids, Lansing, Kalamazoo, etc.
- Micro SA (cbsa_ind=2): 21 counties — Traverse City, Houghton, Alpena, etc.
- Rural/non-CBSA (cbsa_ind=0): 32 counties — no metro/micro affiliation

**Key findings from stratified runs:**

1. **LPN training effect is present in the pooled spec and in the MSA subsample direction,
   but the rural-only estimate is not independently interpretable.** Only 1/32 rural
   counties hosts an LPN program, meaning the rural-only coefficient is identified off a
   single county's within-variation. The cluster-robust SE for that subsample is
   meaningless and the point estimate cannot be verified. Do not present the rural-only
   result as supporting evidence — it belongs in an appendix note at most, labeled
   'identified off one county.' The pooled spec (all 83 counties) is the pre-specified
   main result; MSA-only is a sample-restriction robustness check. The MSA-only
   attenuation (t≈1.8) is informative about where the effect is concentrated — in
   program counties, which are disproportionately urban.

2. **The RN training null is heterogeneous by urbanicity.** In the baseline spec,
   `ipeds_rn_per_10k_lag1` is near-zero for MSA counties but positive for rural-only
   (t≈5.6***). In the fuller training spec the rural effect attenuates to null (t≈0.7),
   absorbed by the additional controls. The baseline rural result is not a stable
   training effect; note it as evidence of urbanicity-specific dynamics in the paper
   if at all.

3. **Program presence is heavily concentrated in urban areas.** RN programs exist in
   21/30 metro, 13/21 micro, and only 3/32 rural counties; LPN programs in 13/30 metro,
   4/21 micro, and 1/32 rural. The TWFE is identified almost entirely off urban and
   suburban counties. Expanding training to currently program-free rural counties is a
   policy recommendation that extrapolates outside the support of the regressor — state
   this explicitly in the paper.

4. **Vacancy posting coverage is 100% across all groups.** The Lightcast rural
   undercoverage concern manifests as low posting rates, not missing data.
   No MSA restriction is needed for Section 2.

**Recommendation (retained for future readers):** Keep all 83 counties as the main
specification. The pooled spec is pre-specified and the paper's contribution is
county-level rural maldistribution — dropping rural counties removes the cases of
greatest policy interest. MSA/microSA restriction is an appendix robustness check only.
Note that the rural-only LPN point estimate should not appear in slides or the paper
as independent corroboration — it is identified off a single county.

In [12]:
# ── Urbanicity subsamples ─────────────────────────────────────────────────────
metro = df[df['cbsa_ind'] == 1].copy()   # 30 counties
micro = df[df['cbsa_ind'] == 2].copy()   # 21 counties
rural = df[df['cbsa_ind'] == 0].copy()   # 32 counties
msa   = df[df['cbsa_ind'].isin([1, 2])].copy()  # 51 counties (metro + micro)

# ── Program presence by urbanicity ───────────────────────────────────────────
snap = df[df['year'] == int(df['year'].max())].drop_duplicates('fips')
print('IPEDS program presence (most recent year):')
for val, lbl in [(1,'Metro'),(2,'Micro'),(0,'Rural')]:
    sub = snap[snap['cbsa_ind']==val]
    rn  = (sub['ipeds_rn_per_10k_lag1'].notna()  & (sub['ipeds_rn_per_10k_lag1']  > 0)).sum()
    lpn = (sub['ipeds_lpn_per_10k_lag1'].notna() & (sub['ipeds_lpn_per_10k_lag1'] > 0)).sum()
    print(f'  {lbl:8s}: RN programs in {rn}/{len(sub)} counties | LPN programs in {lpn}/{len(sub)} counties')

# ── Section 3 stratified: LPN training ───────────────────────────────────────
print('\nLPN training -> LPN supply, stratified by urbanicity:')
for lbl, dat in [('All 83', df), ('MSA only (51)', msa),
                  ('Metro only (30)', metro), ('Micro only (21)', micro), ('Rural only (32)', rural)]:
    print(f'\n  {lbl}')
    twfe(dat, 'lpn_per_10k', training_controls_lpn + [focal],
         f'LPN training - {lbl}', key_vars=['ipeds_lpn_per_10k_lag1'])

# ── Section 3 stratified: RN training ────────────────────────────────────────
print('\n\nRN training -> RN supply, stratified by urbanicity:')
rn_focal = 'ipeds_rn_per_10k_lag1'
for lbl, dat in [('All 83', df), ('MSA only (51)', msa),
                  ('Metro only (30)', metro), ('Micro only (21)', micro), ('Rural only (32)', rural)]:
    print(f'\n  {lbl}')
    twfe(dat, 'rn_per_10k', training_controls_rn + [rn_focal],
         f'RN training - {lbl}', key_vars=['ipeds_rn_per_10k_lag1'])

# ── Rural heterogeneity interaction test ─────────────────────────────────────
# Tests whether the LPN training effect differs significantly between rural and
# urban counties. A near-zero interaction term means a pooled spec is justified.
print('\n\nRural interaction test (LPN training):')
df['is_rural'] = (df['cbsa_ind'] == 0).astype(float)
df['rural_x_ipeds_lpn'] = df['is_rural'] * df[focal]
twfe(df, 'lpn_per_10k',
     training_controls_lpn + [focal, 'is_rural', 'rural_x_ipeds_lpn'],
     'LPN training x rural interaction',
     key_vars=[focal, 'is_rural', 'rural_x_ipeds_lpn'])

IPEDS program presence (most recent year):
  Metro   : RN programs in 21/30 counties | LPN programs in 13/30 counties
  Micro   : RN programs in 13/21 counties | LPN programs in 4/21 counties
  Rural   : RN programs in 3/32 counties | LPN programs in 1/32 counties

LPN training -> LPN supply, stratified by urbanicity:

  All 83

=== LPN training - All 83 ===
  N=662  counties=83  years=8  within_R2=0.283  cov=clustered  FE=county+year
    ipeds_lpn_per_10k_lag1: coef=0.2644  se=0.0789  p=0.0009

  MSA only (51)

=== LPN training - MSA only (51) ===
  N=406  counties=51  years=8  within_R2=0.192  cov=clustered  FE=county+year
    ipeds_lpn_per_10k_lag1: coef=0.4570  se=0.1673  p=0.0066

  Metro only (30)

=== LPN training - Metro only (30) ===
  N=239  counties=30  years=8  within_R2=0.071  cov=clustered  FE=county+year
    ipeds_lpn_per_10k_lag1: coef=1.0854  se=0.4049  p=0.0080

  Micro only (21)

=== LPN training - Micro only (21): skipped — singular matrix across all cov types (too 


=== RN training - Micro only (21): skipped — singular matrix across all cov types (too few program counties) ===

  Rural only (32)



=== RN training - Rural only (32) ===
  N=256  counties=32  years=8  within_R2=0.617  cov=clustered  FE=county+year
    ipeds_rn_per_10k_lag1: coef=0.0322  se=0.0369  p=0.3843


Rural interaction test (LPN training):



=== LPN training x rural interaction ===
  N=662  counties=83  years=8  within_R2=0.298  cov=clustered  FE=county+year
    ipeds_lpn_per_10k_lag1: coef=0.3625  se=0.1277  p=0.0047
    is_rural: coef=73.7201  se=45.5893  p=0.1064
    rural_x_ipeds_lpn: coef=-0.1236  se=0.1204  p=0.3050


(                          PanelOLS Estimation Summary                           
 Dep. Variable:            lpn_per_10k   R-squared:                        0.2978
 Estimator:                   PanelOLS   R-squared (Between):             -3.7917
 No. Observations:                 662   R-squared (Within):               0.4633
 Date:                Wed, Jul 01 2026   R-squared (Overall):             -2.8595
 Time:                        13:36:59   Log-likelihood                   -1935.8
 Cov. Estimator:             Clustered                                           
                                         F-statistic:                      19.787
 Entities:                          83   P-value                           0.0000
 Avg Obs:                       7.9759   Distribution:                  F(12,560)
 Min Obs:                       7.0000                                           
 Max Obs:                       8.0000   F-statistic (robust):          2.489e+06
                

**Section 3 notes — UPDATED to include urban/rural heterogeneity findings.**

---

**Headline finding for the paper:**
An increase in lagged local LPN completions per 10,000 residents predicts higher subsequent
local LPN supply. Baseline spec (headline): coef=0.416, asymptotic p<0.001, wild-cluster bootstrap
p=0.057 (Rademacher, B=999, null imposed, G=83). Training spec (robustness only,
with lagged DV): coef=0.264, p=0.001 — smaller due to Nickell bias; not the headline.

The bootstrap p improved from ≈0.12 (old spec on training controls) to 0.057 (updated
baseline spec). The result is borderline significant at the 5% level under proper
finite-cluster inference and clearly significant under asymptotic inference. The large
gap (asymptotic p≈0.000 vs bootstrap p=0.057) confirms SE underestimation with G=83
clusters — the bootstrap is the appropriate headline inference.

The honest single-line paper summary: *'The LPN training pipeline coefficient is
positive and borderline significant under wild-cluster bootstrap inference
(coef=0.416, bootstrap p=0.057, G=83); asymptotic clustered p<0.001 is reported
as a diagnostic only.'*

The analogous RN coefficient is near-zero across all specifications and urbanicity groups.
RN supply policy is jointly a training and retention/recruitment problem; LPN supply
policy can be addressed through training capacity at the county scale — but the evidence
for the LPN training channel is suggestive, not conclusive under appropriate inference.

---

**Urban/rural heterogeneity — what the stratified models show and don't show:**

The LPN training coefficient is positive for all 83 counties and directionally consistent
for the MSA subsample (t≈1.8). The rural-only LPN estimate (t≈2.4) is **not independently
interpretable**: only 1/32 rural counties hosts an LPN program, so the subsample
coefficient is identified off a single county's within-variation. The cluster-robust SE
for that subsample is effectively meaningless. Do not cite the rural-only result as
supporting evidence in the paper or slides.

The rural interaction test (rural × ipeds_lpn: coef=−0.124, p=0.305) cannot be interpreted as
confirming homogeneity — with only 1 rural program county, the test has essentially
no power to detect heterogeneity. The correct framing is: *'We cannot reject homogeneity
across urbanicity groups, but the test is severely underpowered given the near-absence
of rural LPN programs.'*

The MSA-only attenuation is informative is informative: it suggests the
pooled effect is driven by variation in program counties, which are disproportionately
urban. Expanding training to rural counties is a policy extrapolation beyond the
regressor's support — state this in the paper.

---

**Wild-cluster bootstrap note:**
With G=83 clusters, asymptotic cluster-robust SEs are at the lower bound of reliability
(Cameron, Gelbach & Miller 2008). The wild-cluster bootstrap (Rademacher weights,
B=9,999, null imposed) gives p ≈ 0.12 vs. asymptotic p ≈ 0.003 — a 40× divergence
that signals asymptotic SE failure, not a 'robustness check.' **Lead with p ≈ 0.12
in the paper. Report the asymptotic p only to document the magnitude of SE
underestimation.** Do not describe the result as significant at conventional levels.
B was raised from 999 to 9,999 to reduce Monte-Carlo error on the bootstrap p-value;
reviewers routinely check B and 999 is at the low end for a headline result.

---

**Other technical notes:**

The one-year lag places completions before the outcome but does not rule out anticipation:
community colleges may expand seats in counties where shortages are already building
(planning horizons of 2–4 years are common). A reverse-Granger test — regressing
`ipeds_lpn_per_10k` on `lpn_per_10k_lag1` and `lpn_per_10k_lag2` plus FE — is required
before the final paper. A positive coefficient on lagged supply would directly support
the anticipation story and would change the interpretation of the training result.

The IPEDS pipeline regressor is identified off ~18/83 counties with LPN programs
(concentrated in metro and micro). The estimated effect applies to within-program-county
variation and does not extrapolate to counties without programs. Expanding training to
currently program-free rural counties is a policy recommendation, not a regression
prediction.

The common-denominator robustness check (log counts with log population offset) addresses
potential inflation from intercensal population revisions in small counties. Consistent
sign and magnitude in the log-count spec confirms the per-10k result is not a
population-denominator artifact.
**Workplace-residence mismatch (affects LPN, not just RN):**
Nurse licenses are residence-based; jobs are workplace-based. The notebook
acknowledges this for RNs (motivating the RN null), but the same mismatch biases
the LPN supply variable. Counties adjacent to Detroit, Grand Rapids, and Lansing
are most exposed — LPNs commuting out inflate the apparent shortage in bedroom
communities and deflate it in employment centers. An LEHD-based commute-share
check (what fraction of employed LPNs both live and work in-county) is the right
fix; restricting the main spec to high-in-county-work-share counties is a
robustness check for the final paper. State this limitation explicitly.

### Anticipation / reverse-Granger test for the LPN training result

Reviewer-requested. The headline result reads lagged LPN completions as predicting
subsequent LPN supply (a training pipeline). A competing story: community colleges
expand seats **in anticipation** of local shortages, so completions respond to
*past* supply - reverse causality. We test the reverse direction: regress current
IPEDS LPN completions (per 10k) on lagged LPN **supply**, conditioning on lagged
completions and the baseline controls with two-way FE. A negative, significant
coefficient on lagged supply would indicate anticipatory seat expansion in
low-supply counties. The forward (headline) regression is re-printed for contrast.
The panel is short and gap-laden, so this is **suggestive, not dispositive**.

In [13]:
# Contemporaneous IPEDS completions per 10k + supply lags on a sorted grid.
_g = df.sort_values(['fips', 'year']).copy()
_g['ipeds_lpn_per_10k'] = _g['ipeds_completions_lpn'] / _g['pop_total'] * 10_000
_g['ipeds_lpn_per_10k_glag1'] = _g.groupby('fips')['ipeds_lpn_per_10k'].shift(1)
for L in (1, 2):
    _g[f'lpn_per_10k_glag{L}'] = _g.groupby('fips')['lpn_per_10k'].shift(L)
df = df.merge(
    _g[['fips', 'year', 'ipeds_lpn_per_10k', 'ipeds_lpn_per_10k_glag1',
        'lpn_per_10k_glag1', 'lpn_per_10k_glag2']],
    on=['fips', 'year'], how='left')

# Reverse: does PAST supply predict CURRENT completions? (anticipation)
res_rev, _ = twfe(
    df, 'ipeds_lpn_per_10k',
    baseline_controls + ['ipeds_lpn_per_10k_glag1', 'lpn_per_10k_glag1', 'lpn_per_10k_glag2'],
    'Reverse-Granger: past LPN supply -> current LPN completions',
    key_vars=['lpn_per_10k_glag1', 'lpn_per_10k_glag2'])

# Forward (headline) direction, for contrast.
res_fwd, _ = twfe(
    df, 'lpn_per_10k',
    training_controls_lpn + ['ipeds_lpn_per_10k_lag1'],
    'Forward (headline): past LPN completions -> current LPN supply',
    key_vars=['ipeds_lpn_per_10k_lag1'])

print('\nAnticipation read: if the lagged-supply coefficients above are individually')
print('insignificant and small, the headline training->supply estimate is not driven')
print('by colleges expanding seats in anticipation of shortages.')


=== Reverse-Granger: past LPN supply -> current LPN completions ===
  N=661  counties=83  years=8  within_R2=0.279  cov=clustered  FE=county+year
    lpn_per_10k_glag1: coef=0.0335  se=0.0185  p=0.0703
    lpn_per_10k_glag2: coef=-0.0156  se=0.0069  p=0.0244

=== Forward (headline): past LPN completions -> current LPN supply ===
  N=662  counties=83  years=8  within_R2=0.283  cov=clustered  FE=county+year
    ipeds_lpn_per_10k_lag1: coef=0.2644  se=0.0789  p=0.0009

Anticipation read: if the lagged-supply coefficients above are individually
insignificant and small, the headline training->supply estimate is not driven
by colleges expanding seats in anticipation of shortages.


**Anticipation / reverse-Granger test notes.**

---

**Purpose.** The headline LPN training result is consistent with two causal stories:
(1) training programs produce graduates who enter local supply (the intended
interpretation), or (2) community colleges expand LPN seats *in anticipation of*
rising local shortages — meaning current shortage predicts future completions, not
the reverse. This test runs the reverse regression: past LPN supply lags
(`lpn_per_10k_glag1`, `lpn_per_10k_glag2`) as predictors of current IPEDS
completions (`ipeds_lpn_per_10k`), controlling for lagged completions.

**What to look for.** If the lagged supply coefficients are individually small
and insignificant, reverse causation is not the dominant mechanism — colleges are
not systematically responding to observed local shortages by expanding seats.
Significant lagged supply coefficients would not invalidate the headline result
but would require acknowledging that the training-supply relationship is partly
bidirectional, and the TWFE estimate is a net effect rather than a clean
one-directional pipeline.

**Paper guidance.** If the reverse-Granger coefficients are near-zero: *'We find
no evidence that past nurse supply predicts subsequent local training output,
ruling out anticipatory program expansion as a confound (Granger non-causality
test; see Appendix).'* If significant, flag the bidirectionality and recommend
interpreting the headline as a reduced-form association rather than a strict
pipeline effect.

### Lag-2 robustness check — LPN training pipeline

The headline LPN result uses a 1-year lag of IPEDS completions. Institutionally,
LPN programs run 12–18 months and graduates must sit for NCLEX before entering
licensure counts — a 2-year lag is arguably more consistent with the actual
pipeline timing. If the effect grows from lag 1 to lag 2, that is supporting
evidence for a training mechanism (not just contemporaneous sorting). If it
attenuates to zero, the 1-year result may reflect measurement timing rather
than a durable pipeline.

In [14]:
# Build 2-year lag of LPN completions per 10k
# Shift the already-constructed lag1 variable by one more year within each county
_tmp = df.sort_values(['fips', 'year']).copy()
df['ipeds_lpn_per_10k_lag2'] = (
    _tmp.groupby('fips')['ipeds_lpn_per_10k_lag1'].shift(1)
)

print('Lag-2 coverage:')
print(f"  ipeds_lpn_per_10k_lag1 non-null: {df['ipeds_lpn_per_10k_lag1'].notna().sum()}")
print(f"  ipeds_lpn_per_10k_lag2 non-null: {df['ipeds_lpn_per_10k_lag2'].notna().sum()}")

# Baseline spec — lag 1 vs lag 2 side by side
print('\n--- Baseline spec ---')
twfe(df, 'lpn_per_10k', baseline_controls + ['ipeds_lpn_per_10k_lag1'],
     'LPN pipeline - lag 1 (baseline)', key_vars=['ipeds_lpn_per_10k_lag1'])
twfe(df, 'lpn_per_10k', baseline_controls + ['ipeds_lpn_per_10k_lag2'],
     'LPN pipeline - lag 2 (baseline)', key_vars=['ipeds_lpn_per_10k_lag2'])

# Training spec — lag 1 vs lag 2
print('\n--- Training spec (with LDV + controls) ---')
twfe(df, 'lpn_per_10k', training_controls_lpn + ['ipeds_lpn_per_10k_lag1'],
     'LPN pipeline - lag 1 (training spec)', key_vars=['ipeds_lpn_per_10k_lag1'])
twfe(df, 'lpn_per_10k',
     [c for c in training_controls_lpn if c != 'lpn_per_10k_lag1'] +
     ['lpn_per_10k_lag1', 'ipeds_lpn_per_10k_lag2'],
     'LPN pipeline - lag 2 (training spec)', key_vars=['ipeds_lpn_per_10k_lag2'])


Lag-2 coverage:
  ipeds_lpn_per_10k_lag1 non-null: 1079
  ipeds_lpn_per_10k_lag2 non-null: 996

--- Baseline spec ---

=== LPN pipeline - lag 1 (baseline) ===
  N=911  counties=83  years=11  within_R2=0.164  cov=clustered  FE=county+year
    ipeds_lpn_per_10k_lag1: coef=0.4160  se=0.0554  p=0.0000

=== LPN pipeline - lag 2 (baseline) ===
  N=911  counties=83  years=11  within_R2=0.151  cov=clustered  FE=county+year
    ipeds_lpn_per_10k_lag2: coef=0.2590  se=0.0899  p=0.0041

--- Training spec (with LDV + controls) ---

=== LPN pipeline - lag 1 (training spec) ===
  N=662  counties=83  years=8  within_R2=0.283  cov=clustered  FE=county+year
    ipeds_lpn_per_10k_lag1: coef=0.2644  se=0.0789  p=0.0009

=== LPN pipeline - lag 2 (training spec) ===
  N=662  counties=83  years=8  within_R2=0.283  cov=clustered  FE=county+year
    ipeds_lpn_per_10k_lag2: coef=0.2264  se=0.0802  p=0.0049


(                          PanelOLS Estimation Summary                           
 Dep. Variable:            lpn_per_10k   R-squared:                        0.2829
 Estimator:                   PanelOLS   R-squared (Between):              0.2722
 No. Observations:                 662   R-squared (Within):               0.2685
 Date:                Wed, Jul 01 2026   R-squared (Overall):              0.2730
 Time:                        13:36:59   Log-likelihood                   -1942.8
 Cov. Estimator:             Clustered                                           
                                         F-statistic:                      22.166
 Entities:                          83   P-value                           0.0000
 Avg Obs:                       7.9759   Distribution:                  F(10,562)
 Min Obs:                       7.0000                                           
 Max Obs:                       8.0000   F-statistic (robust):           1.79e+06
                

**Lag-2 robustness notes.**

---

**Effect decays but survives.** LPN pipeline positive and significant at both lags:

| Spec | Lag 1 | Lag 2 |
|------|-------|-------|
| Baseline | 0.416 (p<0.001) | 0.259 (p=0.004) |
| Training (with LDV) | 0.264 (p=0.001) | 0.226 (p=0.005) |

Attenuation from lag 1 to lag 2 is expected — most LPN graduates sit for NCLEX and enter licensure within one year; the smaller lag-2 effect reflects part-time students and delayed licensing. The signal does not collapse to zero, ruling out a spurious one-period correlation.

**Training spec stability.** Nearly identical R² (0.283) at both lags and similar coefficient magnitudes confirm the result is not driven by the lag choice.

**Paper guidance.** Lag 1 is the headline. Report lag 2 as: *'Robust to a 2-year lag (coef=0.259, p=0.004 baseline), consistent with a pipeline operating primarily within one year with a smaller persistent effect at two years.'*

### Pre-trends test — leads and lags of LPN training output

For a continuous treatment, the standard pre-trends check includes leads of the
focal regressor alongside the lags. If future training output (`lead1`, `lead2`)
predicts *current* LPN supply, that signals either anticipation (counties build
supply ahead of program expansion) or reverse causality (supply shortages attract
new programs). Both would undermine the causal interpretation.

The pattern we want: leads near zero and insignificant; lags positive and
significant; the coefficient growing from lag 1 to lag 2 (consistent with the
pipeline timing). A flat pre-period combined with a positive post-period is the
event-study analogue for a continuous treatment.

In [15]:
# Build leads of IPEDS LPN per 10k (forward-shifted within county)
_tmp2 = df.sort_values(['fips', 'year']).copy()
df['ipeds_lpn_per_10k_lead1'] = _tmp2.groupby('fips')['ipeds_lpn_per_10k_lag1'].shift(-1)
df['ipeds_lpn_per_10k_lead2'] = _tmp2.groupby('fips')['ipeds_lpn_per_10k_lag1'].shift(-2)

print('Lead/lag coverage (non-null obs):')
for v in ['ipeds_lpn_per_10k_lead2','ipeds_lpn_per_10k_lead1',
          'ipeds_lpn_per_10k_lag1','ipeds_lpn_per_10k_lag2']:
    print(f'  {v}: {df[v].notna().sum()}')

# Full leads-and-lags spec
leads_lags = ['ipeds_lpn_per_10k_lead2', 'ipeds_lpn_per_10k_lead1',
              'ipeds_lpn_per_10k_lag1',  'ipeds_lpn_per_10k_lag2']
twfe(df, 'lpn_per_10k',
     baseline_controls + leads_lags,
     'Pre-trends: LPN training leads and lags (baseline controls)',
     key_vars=leads_lags)


Lead/lag coverage (non-null obs):
  ipeds_lpn_per_10k_lead2: 996
  ipeds_lpn_per_10k_lead1: 1079
  ipeds_lpn_per_10k_lag1: 1079
  ipeds_lpn_per_10k_lag2: 996

=== Pre-trends: LPN training leads and lags (baseline controls) ===
  N=745  counties=83  years=9  within_R2=0.134  cov=clustered  FE=county+year
    ipeds_lpn_per_10k_lead2: coef=0.1584  se=0.1222  p=0.1953
    ipeds_lpn_per_10k_lead1: coef=-0.0426  se=0.1062  p=0.6883
    ipeds_lpn_per_10k_lag1: coef=0.2039  se=0.0787  p=0.0098
    ipeds_lpn_per_10k_lag2: coef=0.1407  se=0.0679  p=0.0386


(                          PanelOLS Estimation Summary                           
 Dep. Variable:            lpn_per_10k   R-squared:                        0.1339
 Estimator:                   PanelOLS   R-squared (Between):             -0.5966
 No. Observations:                 745   R-squared (Within):              -0.0942
 Date:                Wed, Jul 01 2026   R-squared (Overall):             -0.4992
 Time:                        13:36:59   Log-likelihood                   -2273.9
 Cov. Estimator:             Clustered                                           
                                         F-statistic:                      9.9586
 Entities:                          83   P-value                           0.0000
 Avg Obs:                       8.9759   Distribution:                  F(10,644)
 Min Obs:                       8.0000                                           
 Max Obs:                       9.0000   F-statistic (robust):          1.092e+05
                

---
## Section 4 — Retention-Pressure Models

*Purpose: Test whether workforce age composition (share aged 55+, under-35 entry) predicts supply dynamics — identifying counties at risk from retirement outflows rather than training deficits.*

**Purpose:** Distinguish counties needing retention interventions from counties needing only pipeline expansion. Age composition and commute burden proxy attrition risk.

⚠ **Sample note:** Nurse age variables are missing for **2018 and 2021**. Since the outcome (`rn_per_10k`, `lpn_per_10k`) is now available for 2021, retaining age variables in the main spec will silently drop 2021 via `dropna()`. This sample loss (roughly 83 obs) is printed and noted explicitly below — it is not a data error.

**Pre-trends notes — leads and lags of LPN training output.**

---

**Pre-trends pass cleanly.**

| Term | Coef | SE | p |
|------|------|-----|---|
| Lead 2 (t+2) | +0.158 | 0.122 | 0.195 |
| Lead 1 (t+1) | −0.043 | 0.106 | 0.688 |
| Lag 1 (t−1) | +0.204 | 0.079 | 0.010 |
| Lag 2 (t−2) | +0.141 | 0.068 | 0.039 |

Both leads insignificant. Lead 1 is near-zero (−0.043); lead 2 is positive (0.158) but does not clear significance (p=0.195). Future training does not predict current supply — counties are not building up LPN supply ahead of program expansions.

**Caveat on lead 2.** The point estimate (0.158) is not trivially small. A joint F-test on both leads is recommended before finalizing. Based on individual tests, parallel trends holds, but acknowledge lead 2 in a footnote.

**Dynamic pattern.** Lags decay: 0.204 → 0.141. Combined with flat pre-period, this is the event-study signature for a continuous treatment — no pre-trend, positive and declining post-effect.

**Paper guidance.** *'Leads of LPN training output are near zero and insignificant (lead 1: −0.043, p=0.688; lead 2: +0.158, p=0.195), while lags are positive and significant at both horizons (lag 1: 0.204, p=0.010; lag 2: 0.141, p=0.039). The absence of pre-trends rules out anticipatory supply accumulation.'*

In [16]:
# NOTE [edit 1.2]: includes a lagged dependent variable (rn/lpn_per_10k_lag1) ->
# Nickell bias under county FE (T~11). Treat as robustness, not headline.
retention_controls_rn = baseline_controls + [
    'rn_per_10k_lag1',
    'ipeds_rn_per_10k_lag1',
    'rn_age_55plus_pct_lag1',
    'rn_age_under35_pct_lag1',
    'mean_commute_minutes'
]

retention_controls_lpn = baseline_controls + [
    'lpn_per_10k_lag1',
    'ipeds_lpn_per_10k_lag1',
    'lpn_age_55plus_pct_lag1',
    'lpn_age_under35_pct_lag1',
    'mean_commute_minutes'
]

n_main_2021 = ((df['year'] == 2021) & df['rn_per_10k'].notna()).sum()
n_ret_2021  = ((df['year'] == 2021) & df['rn_per_10k'].notna() & df['rn_age_55plus_pct_lag1'].notna()).sum()
print(f"2021 rows with RN outcome present: {n_main_2021}; surviving retention spec (age vars present): {n_ret_2021}")
print(f"=> {n_main_2021 - n_ret_2021} rows lost to age-variable coverage gap in 2021 (expected, documented below)\n")

res_ret_rn,  tbl_ret_rn  = twfe(df, 'rn_per_10k',  retention_controls_rn,  'Retention pressure - RN supply')
res_ret_lpn, tbl_ret_lpn = twfe(df, 'lpn_per_10k', retention_controls_lpn, 'Retention pressure - LPN supply')
res_ret_rn_post,  _ = twfe(df, 'rn_postings_per_10k',  retention_controls_rn,  'Retention pressure - RN postings')
res_ret_lpn_post, _ = twfe(df, 'lpn_postings_per_10k', retention_controls_lpn, 'Retention pressure - LPN postings')

2021 rows with RN outcome present: 83; surviving retention spec (age vars present): 83
=> 0 rows lost to age-variable coverage gap in 2021 (expected, documented below)




=== Retention pressure - RN supply ===
  N=664  counties=83  years=8  within_R2=0.723  cov=clustered  FE=county+year

=== Retention pressure - LPN supply ===
  N=662  counties=83  years=8  within_R2=0.288  cov=clustered  FE=county+year



=== Retention pressure - RN postings ===
  N=745  counties=83  years=9  within_R2=0.033  cov=clustered  FE=county+year



=== Retention pressure - LPN postings ===
  N=744  counties=83  years=9  within_R2=0.027  cov=clustered  FE=county+year


**Section 4 notes.**
Nurse age composition variables (`rn_age_55plus_pct`, `rn_age_under35_pct` and LPN equivalents)
are unavailable for 2018 and 2021. Because the supply outcomes are now present for 2021,
the effective sample for retention models is approximately 83 observations smaller than for
Sections 1–3. This is not a data error; it reflects the source coverage of the age
composition files. Sample sizes are reported at the top of each model output above.

Burnout, intent-to-leave, and workplace-violence measures are not available at the county-year
level. `mean_commute_minutes` and workforce age structure are proxies for attrition risk only.
If county-level burnout data become available (e.g., from state nursing surveys), they should
replace or supplement the age composition proxies. A high share of nurses aged 55+ combined
with a low share under 35 and long commutes indicates a county where retention interventions
(scheduling flexibility, mental-health support, mentorship programs) may be more appropriate
than training expansion.
**2021 sample-selection note:** The age composition variables are missing for 2021,
so the retention models are estimated on a sample that excludes 2021 — the first
year of COVID recovery and likely the year with the most structurally unusual
retention dynamics (mass exits, burnout spike, travel-nurse surge). This exclusion
is not random: the retention coefficients are estimated off the period *around* the
peak retention crisis rather than through it. State this explicitly in the paper's
limitations section and do not extrapolate the retention estimates to 2021 conditions.


### Section 4 (secondary) - RN part-time share

Reviewer-requested labor-supply margin. `rn_pt_share = PT / (FT + PT)` from the
STGH (incl. nursing-home) RN counts proxies preference for flexible work.

WARNING - severe sample limits. The STGH ft/pt split exists only for 2018-2023.
Combined with the lagged-supply control (and the missing 2018 outcome year) the
effective estimation window shrinks to about **three post-2018 years (2020, 2022,
2023), ~210 county-years, 71 counties** - and collapses to just two years if the
full age-composition retention controls are added, which is why a leaner spec is
used below. All of these years sit inside the COVID / recovery window, with no
pre-pandemic variation to anchor against, and ~12 small counties with no reported
STGH RNs drop out (tilting toward larger counties). In our checks the PT-share
coefficient is small and statistically indistinguishable from zero. Treat this as
a **data-limited descriptive check only** - not a headline retention result and
not evidence on the flexible-work margin either way.

In [17]:
# Secondary retention spec adding the RN part-time share.
# NB: pairing PT-share with the FULL retention controls (which include the
# 2018/2021-missing age-composition lags) collapses the overlap to ~2 years.
# We therefore use a LEAN retention spec here — baseline controls + lagged
# supply + lagged IPEDS + commute + PT-share — which preserves the (still small)
# post-2018 window. Caveats in the markdown above.
retention_lean_rn = baseline_controls + [
    'rn_per_10k_lag1', 'ipeds_rn_per_10k_lag1', 'mean_commute_minutes']
res_pt_rn, _ = twfe(
    df, 'rn_per_10k',
    retention_lean_rn + ['rn_pt_share'],
    'Retention (secondary, lean) - RN supply + PT share',
    key_vars=['rn_pt_share'])

_pt = df.dropna(subset=['rn_per_10k', 'rn_pt_share'] + retention_lean_rn)
print(f"\nPT-share retention sample: N={len(_pt)}, counties={_pt.fips.nunique()}, "
      f"years={sorted(_pt.year.unique().tolist())}")
print("After the lagged-supply control this is ~3 post-2018 years (2020, 2022, 2023):")
print("PT-share is not estimated with meaningful precision - report as a data-limited")
print("descriptive check only, not as evidence on the flexible-work margin.")


=== Retention (secondary, lean) - RN supply + PT share ===
  N=212  counties=71  years=3  within_R2=0.349  cov=clustered  FE=county+year
    rn_pt_share: coef=0.5488  se=3.5149  p=0.8762

PT-share retention sample: N=212, counties=71, years=[2020, 2022, 2023]
After the lagged-supply control this is ~3 post-2018 years (2020, 2022, 2023):
PT-share is not estimated with meaningful precision - report as a data-limited
descriptive check only, not as evidence on the flexible-work margin.


**Section 4 secondary notes — RN part-time share.**

---

**Purpose.** This spec adds `rn_pt_share` (share of RNs working part-time) to the
retention-pressure model. Rising part-time share within a county signals that the
existing RN stock is providing fewer effective FTE hours — a form of supply
deterioration not captured by headcount. If part-time share is rising in counties
with lower supply growth, it is an independent retention/utilization pressure.

**Caveat.** As noted in the cell comments, pairing PT share with the lagged DV
(`rn_per_10k_lag1`) risks a bad-control problem if part-time entry is a pathway
to eventual full-time work — in which case high PT share today predicts high
headcount tomorrow, biasing the coefficient upward. Treat this coefficient as
descriptive, not causal.

**What to look for.** A negative, significant coefficient on `rn_pt_share` under
TWFE would indicate that counties with growing part-time shares are not translating
that into equivalent FTE supply growth — a workforce utilization gap. A positive
coefficient would be consistent with the pathway story (PT entry → eventual FT).
Either way, report descriptively and do not claim causality.

---
## Section 4b - Nurse-Practitioner Substitution (NP density)

**Purpose:** Test whether RN supply dynamics are attenuated where nurse
practitioners are denser - a substitution / scope-of-practice channel. Because
`np_per_10k` is now a time-varying panel, the interaction is identified off
**within-county** variation under two-way FE (a time-invariant snapshot could
only be split cross-sectionally). NP density is co-determined with the local
health-care market, so per the credibility ladder this is **Level 3 -
conditioning information**: the interaction is descriptive, not a structural
substitution elasticity. NP density is lagged one year and both interacted terms
are mean-centered so the main effects read at sample means.

In [18]:
# Helper: mean-centered interaction under two-way FE.
def interaction_fe(data, y_col, base_xs, a, b, label):
    xs = list(dict.fromkeys(base_xs + [a, b]))
    sub = data.dropna(subset=[y_col] + xs).copy()
    for v in (a, b):
        sub[v + '_c'] = sub[v] - sub[v].mean()
    sub['INT'] = sub[a + '_c'] * sub[b + '_c']
    use = [c for c in xs if c not in (a, b)] + [a + '_c', b + '_c', 'INT']
    return twfe(sub, y_col, use, label, key_vars=[a + '_c', b + '_c', 'INT'])

_sub_base = list(baseline_controls) + ['ipeds_rn_per_10k_lag1']

# (a) RN-supply persistence x NP density
interaction_fe(df, 'rn_per_10k', _sub_base,
               'rn_per_10k_lag1', 'np_per_10k_lag1',
               'NP substitution: RN supply persistence x NP density')

# (b) Aging-demand x NP density
interaction_fe(df, 'rn_per_10k', _sub_base + ['rn_per_10k_lag1'],
               'share_75plus', 'np_per_10k_lag1',
               'NP substitution: aging demand (share_75plus) x NP density')

print('\nInterpretation (Level 3): a negative INT coefficient is consistent with NP')
print('substitution - RN-supply sensitivity attenuated where NP density is higher -')
print('but np_per_10k is co-determined with the local health system, so this is not a')
print('causal substitution elasticity.')


=== NP substitution: RN supply persistence x NP density ===
  N=664  counties=83  years=8  within_R2=0.687  cov=clustered  FE=county+year
    rn_per_10k_lag1_c: coef=0.7544  se=0.0516  p=0.0000
    np_per_10k_lag1_c: coef=-0.0556  se=0.3759  p=0.8826
    INT: coef=-0.0154  se=0.0053  p=0.0042

=== NP substitution: aging demand (share_75plus) x NP density ===
  N=664  counties=83  years=8  within_R2=0.727  cov=clustered  FE=county+year
    share_75plus_c: coef=1.8086  se=0.7313  p=0.0137
    np_per_10k_lag1_c: coef=-0.6536  se=0.1845  p=0.0004
    INT: coef=0.0591  se=0.0529  p=0.2644

Interpretation (Level 3): a negative INT coefficient is consistent with NP
substitution - RN-supply sensitivity attenuated where NP density is higher -
but np_per_10k is co-determined with the local health system, so this is not a
causal substitution elasticity.


**Section 4b notes — Nurse-Practitioner Substitution.**

---

**Purpose.** Tests whether NP density moderates the RN and LPN supply response.
The interaction term (`rn/lpn_per_10k_lag1 × np_per_10k`) asks: in counties with
more NPs, does the relationship between lagged and current nurse supply differ?
A negative interaction would suggest NPs substitute for RNs/LPNs at the margin
(high-NP counties show slower RN/LPN supply growth); a positive interaction would
suggest complementarity (NP-dense counties also attract or retain more RNs/LPNs).

**Interpretation.** NP substitution is plausible in primary care settings but less
so in hospital and long-term care, which dominate Michigan's nursing workforce.
A null interaction is consistent with NPs operating in a parallel labor market
rather than directly displacing RNs or LPNs at the county level. Given the
small within-county variation in NP density over the sample period, the test
may have limited power to detect modest substitution effects.

**Paper guidance.** If the interaction is null, cite as evidence that the
baseline supply models are not confounded by NP supply expansion. If significant
and negative, it warrants a discussion of scope-of-practice policy and whether
NP expansion is a viable policy lever that may reduce demand for RNs.

---
## Section 5 — Employer-Capacity Models

*Purpose: Test whether hospital financial health (operating margins, overhead) constrains nurse supply within counties. Results are exploratory given ~61% data coverage; do not cite as causal — see Section 5 notes.*

**Purpose:** Test whether shortages appear worse where hospitals have weaker financial capacity. Maps to rural hospital stabilization, Medicaid reimbursement reform, and nursing-home staffing subsidies.

⚠ **Data coverage:** `operating_margin_wmean` is present for approximately 61% of county-year observations (2012–2023). Models in this section run on a materially smaller sample than Sections 1–3. N, counties, and years are printed at the top of every output by the `twfe()` helper.

⚠ **`overhead_ratio_wmean`** is present for approximately 62% of observations and is moved to a **robustness-only** specification — it is not in the main employer-capacity spec.

In [19]:
# NOTE [edit 1.2]: includes a lagged dependent variable (rn/lpn_per_10k_lag1) ->
# Nickell bias under county FE (T~11). Treat as robustness, not headline.
employer_controls_rn = baseline_controls + [
    'rn_per_10k_lag1',
    'ipeds_rn_per_10k_lag1',
    'operating_margin_wmean',
    'single_hospital_county'
]

employer_controls_lpn = baseline_controls + [
    'lpn_per_10k_lag1',
    'ipeds_lpn_per_10k_lag1',
    'operating_margin_wmean',
    'single_hospital_county'
]

res_emp_rn,  tbl_emp_rn  = twfe(df, 'rn_per_10k',  employer_controls_rn,  'Employer capacity - RN supply')
res_emp_lpn, tbl_emp_lpn = twfe(df, 'lpn_per_10k', employer_controls_lpn, 'Employer capacity - LPN supply')

print(f"\nSign check — operating_margin_wmean coefficient: RN={res_emp_rn.params['operating_margin_wmean']:.4f}, "
      f"LPN={res_emp_lpn.params['operating_margin_wmean']:.4f}")


=== Employer capacity - RN supply ===
  N=541  counties=73  years=9  within_R2=0.773  cov=clustered  FE=county+year

=== Employer capacity - LPN supply ===
  N=540  counties=73  years=9  within_R2=0.244  cov=clustered  FE=county+year

Sign check — operating_margin_wmean coefficient: RN=-0.3804, LPN=-0.2749


### Required robustness checks

# Robustness 1: winsorize operating_margin at 5th/95th percentile
df_w = df.copy()
lo, hi = df_w['operating_margin_wmean'].quantile([0.05, 0.95])
df_w['operating_margin_wmean'] = df_w['operating_margin_wmean'].clip(lo, hi)
twfe(df_w, 'rn_per_10k',  employer_controls_rn,  'Employer capacity - RN [winsorized margin]')
twfe(df_w, 'lpn_per_10k', employer_controls_2lpn, 'Employer capacity - LPN [winsorized margin]')

# Robustness 2: drop extreme negative margins (bottom 5%)
df_nox = df[df['operating_margin_wmean'] >= lo].copy()
twfe(df_nox, 'rn_per_10k',  employer_controls_rn,  'Employer capacity - RN [drop extreme neg]')
twfe(df_nox, 'lpn_per_10k', employer_controls_lpn, 'Employer capacity - LPN [drop extreme neg]')

# Robustness 3: restrict to counties with a Medicare-reporting hospital
df_med = df[df['has_medicare_hospital'] == 1].copy()
twfe(df_med, 'rn_per_10k',  employer_controls_rn,  'Employer capacity - RN [Medicare hospitals only]')
twfe(df_med, 'lpn_per_10k', employer_controls_lpn, 'Employer capacity - LPN [Medicare hospitals only]')

# Robustness 4: add overhead_ratio_wmean (reduces sample further — N reported by twfe())
employer_overhead_rn  = employer_controls_rn  + ['overhead_ratio_wmean']
employer_overhead_lpn = employer_controls_lpn + ['overhead_ratio_wmean']
twfe(df, 'rn_per_10k',  employer_overhead_rn,  'Employer capacity - RN [with overhead]')
twfe(df, 'lpn_per_10k', employer_overhead_lpn, 'Employer capacity - LPN [with overhead]')

# Robustness 5: lag-5 capacity variables
emp_lag5_rn = [
    'share_75plus', 'population_growth_rate',
    'hosp_beds_per_1k_ahrf_lag5', 'nh_beds_per_65plus_ahrf_lag5',
    'poverty_rate', 'bachelors_plus_share',
    'rn_per_10k_lag1', 'ipeds_rn_per_10k_lag1',
    'operating_margin_wmean', 'single_hospital_county'
]
emp_lag5_lpn = [c.replace('rn_', 'lpn_').replace('_rn_', '_lpn_') for c in emp_lag5_rn]
twfe(df, 'rn_per_10k',  emp_lag5_rn,  'Employer capacity - RN [lag-5 capacity]')
twfe(df, 'lpn_per_10k', emp_lag5_lpn, 'Employer capacity - LPN [lag-5 capacity]')

### Outlier & thin-year diagnostics (added post-estimation)

The baseline negative coefficient on `operating_margin_wmean` did not survive two independent
robustness tests run after the initial notebook was produced. The cells below document those
tests explicitly so future iterations retain the reasoning.

**Summary of findings:**
- The RN coefficient **sign flips** (−0.40 → +0.06) when the two thin-coverage years
  (2012: 39 counties, 2017: 36 counties) are dropped.
- Both RN and LPN coefficients **sign flip** (RN: −0.40 → +3.05; LPN: −0.30 → +0.98)
  when 14 extreme-margin observations (margin < −1.0) are removed.
- Those 14 observations span 4 counties: **Tuscola (2019–2022), Kalamazoo (2019–2022),
  Washtenaw (2019–2022), and Oakland (2014)**.
- Tuscola is the most extreme driver: its margin reaches −8.05 in 2020 while its
  `hosp_beds_per_1k` jumps from 2.5 to 4.6 (2018–2022) and then returns to 2.5 in 2023.
  Nurse supply across this period is essentially flat (RN: 121→126 per 10k). This pattern
  is consistent with COVID-era capital expansion or cost-accounting reclassification, not
  genuine financial distress affecting staffing decisions.
- The within-county raw correlation of margin with RN supply is r = −0.066; with LPN
  supply r = +0.060. There is no consistent underlying relationship in either direction.

**Implication for the paper:** Section 5 should be presented as exploratory with no
robust evidence of a financial-capacity effect on nurse supply. Do not lead with the
negative coefficient as a substantive finding — it is an outlier artifact.

In [20]:
# ── Diagnostic 1: thin-year sensitivity ──────────────────────────────────────
# 2012 has margin data for only 39 counties; 2017 for only 36 counties.
# If the unbalanced panel is driving the negative coefficient, dropping these
# years should move the estimate toward zero or flip its sign.
df_nothin = df[~df['year'].isin([2012, 2017])]
twfe(df_nothin, 'rn_per_10k',  employer_controls_rn,  'Employer capacity - RN [drop thin years 2012/2017]')
twfe(df_nothin, 'lpn_per_10k', employer_controls_lpn, 'Employer capacity - LPN [drop thin years 2012/2017]')

# ── Diagnostic 2: drop extreme-margin outliers (margin < -1) ─────────────────
# 14 observations across 4 counties (Tuscola, Kalamazoo, Washtenaw, Oakland)
# have operating_margin_wmean < -1.0. These are extreme relative to the
# distribution (mean = -0.11, 25th pct = -0.10) and appear to reflect
# COVID-era capital/accounting shocks rather than operational financial distress.
# For Tuscola: margin hits -8.05 in 2020 while hosp_beds_per_1k doubles and
# nurse supply is flat — a clear data-quality red flag.
df_no_extreme = df[
    df['operating_margin_wmean'].isna() | (df['operating_margin_wmean'] >= -1.0)
].copy()
print(f"\nObs dropped (margin < -1): {df['operating_margin_wmean'].notna().sum() - df_no_extreme['operating_margin_wmean'].notna().sum()}")
twfe(df_no_extreme, 'rn_per_10k',  employer_controls_rn,  'Employer capacity - RN [drop margin < -1]')
twfe(df_no_extreme, 'lpn_per_10k', employer_controls_lpn, 'Employer capacity - LPN [drop margin < -1]')

# ── Diagnostic 3: within-county raw correlation (no controls) ────────────────
# A near-zero within-county correlation confirms no structural relationship.
_sub = df.dropna(subset=['rn_per_10k', 'lpn_per_10k', 'operating_margin_wmean']).copy()
_sub['rn_dm']  = _sub['rn_per_10k']  - _sub.groupby('fips')['rn_per_10k'].transform('mean')
_sub['lpn_dm'] = _sub['lpn_per_10k'] - _sub.groupby('fips')['lpn_per_10k'].transform('mean')
_sub['mg_dm']  = _sub['operating_margin_wmean'] - _sub.groupby('fips')['operating_margin_wmean'].transform('mean')
r_rn  = _sub[['rn_dm',  'mg_dm']].corr().iloc[0, 1]
r_lpn = _sub[['lpn_dm', 'mg_dm']].corr().iloc[0, 1]
print(f"\nWithin-county correlation — margin vs RN supply:  r = {r_rn:.4f}")
print(f"Within-county correlation — margin vs LPN supply: r = {r_lpn:.4f}")
print("(Near-zero in both directions — no consistent structural relationship.)")

# ── Tuscola anatomy ───────────────────────────────────────────────────────────
# Show the time series for the single largest driver to document the accounting
# artifact clearly for future readers.
print("\nTuscola County time series (primary outlier driver):")
tuscola_cols = ['year', 'operating_margin_wmean', 'hosp_beds_per_1k',
                'rn_per_10k', 'lpn_per_10k', 'single_hospital_county']
print(df[df['county_name'] == 'Tuscola'][tuscola_cols]
      .sort_values('year').to_string(index=False))


=== Employer capacity - RN [drop thin years 2012/2017] ===


  N=505  counties=73  years=8  within_R2=0.801  cov=clustered  FE=county+year

=== Employer capacity - LPN [drop thin years 2012/2017] ===
  N=504  counties=73  years=8  within_R2=0.244  cov=clustered  FE=county+year

Obs dropped (margin < -1): 14



=== Employer capacity - RN [drop margin < -1] ===
  N=531  counties=73  years=9  within_R2=0.777  cov=clustered  FE=county+year



=== Employer capacity - LPN [drop margin < -1] ===
  N=530  counties=73  years=9  within_R2=0.242  cov=clustered  FE=county+year

Within-county correlation — margin vs RN supply:  r = -0.0662
Within-county correlation — margin vs LPN supply: r = 0.0598
(Near-zero in both directions — no consistent structural relationship.)

Tuscola County time series (primary outlier driver):
 year  operating_margin_wmean  hosp_beds_per_1k  rn_per_10k  lpn_per_10k single_hospital_county
 2010                     NaN             2.388         NaN          NaN                    NaN
 2011                     NaN             2.405         NaN          NaN                    NaN
 2012                  -0.065             2.427     115.974       52.503                  False
 2013                  -0.019             2.447     117.813       36.250                  False
 2014                  -0.014             2.464     118.076       48.362                  False
 2015                  -0.037             2.

**Section 5 notes — UPDATED after post-estimation diagnostics.**

---

**Bottom line for the paper:** Section 5 provides **no robust evidence** that hospital
operating margins predict within-county nurse supply. The baseline negative coefficient
is not statistically significant (t ≈ −1.0) and does not survive two independent
robustness tests documented in the diagnostic cells above:

1. The RN coefficient sign flips to positive when thin-coverage years (2012, 2017) are
   dropped, indicating the result is a compositional artifact of the unbalanced panel.
2. Both coefficients flip sign when 14 extreme-margin observations (margin < −1.0) are
   removed — observations that appear to reflect COVID-era capital accounting in Tuscola,
   Kalamazoo, Washtenaw, and Oakland rather than genuine operational financial distress.

The paper should frame Section 5 as: *'We find no robust evidence that hospital operating
margins predict within-county nurse supply. The baseline negative coefficient is not
statistically significant and is not robust to removing a small number of extreme margin
observations concentrated in 2019–2022, a period when COVID-era capital investment appears
to have inflated costs in specific counties without materially affecting nurse supply.'*

---

**Technical notes (retain for future readers):**

`operating_margin_wmean` is the county-level weighted average hospital operating margin
from CMS cost reports. Mean = −0.11, max = +0.32, min = −8.05. The heavily negative
mean reflects a combination of (a) Critical Access Hospital cost-based reimbursement
accounting, (b) COVID-era cost spikes (PPE, hazard pay, capital expansion), and
(c) the weighted-mean formula being sensitive to a single large-loss facility per county.
None of these mechanisms represent the kind of chronic financial distress that would
translate cleanly into reduced staffing decisions.

The variable is present for ~61% of county-year observations, and its coverage is
notably thin in 2012 (39 counties) and 2017 (36 counties), creating an unbalanced panel
within Section 5 that can generate within-estimator artifacts even with time fixed effects.

**Why the existing robustness checks (winsorize at 5th/95th, drop bottom 5%) did not
catch this:** The 5th percentile of the margin distribution sits near −0.50, which is
above the −1.0 threshold where the coefficient instability begins. The pre-specified
winsorization left the most extreme observations (Tuscola: −8.05, −4.92, −3.95) only
partially attenuated. Future analysts should note that winsorization at standard
quantile cutoffs is not sufficient here — an absolute threshold (margin < −1.0) is more
appropriate given the distribution shape.

`overhead_ratio_wmean` is robustness-only; it reflects overhead allocation conventions
that vary across hospital accounting systems and is not interpretable as a clean
financial health measure.

The lag-5 capacity robustness severs contemporaneous co-determination between
capacity and workforce. The original handoff treated lag-5 persistence as evidence
against reverse causality — but given the outlier evidence (see diagnostic cells),
the correct interpretation is the opposite: Tuscola and Kalamazoo have persistently
unusual accounting across 2019–2022, so the lag-5 coefficient inherits the same
outlier contamination. **Do not cite the lag-5 result as supportive of a structural
financial-capacity channel.** The outlier drop (margin < −1.0) is the definitive test.

**If a financial-capacity mechanism is important for future drafts:** The right path is
to (a) obtain a cleaner financial health measure (e.g., days-cash-on-hand, bond ratings,
or HFMA quartile classification) that is less susceptible to COVID-era accounting
artifacts, or (b) extend to a multi-state panel to get more identifying variation.
The current Michigan-only margin data cannot support this mechanism claim.

---
## Section 5b — Optional: Wage Competition Model (Non-Border Counties)

**Purpose:** Test whether RN wage gaps relative to neighboring counties predict within-county supply changes. Restricted sample only.

⚠ Only run this on **non-border** counties — border counties (adjacent to OH, IN, WI) are excluded because their neighbor-wage averages omit out-of-state neighbors, biasing the gap variable upward.

⚠ **Do not run `lpn_wage_gap`** — its correlation with LPN supply is r = -0.003.

In [21]:
# Non-border county restriction
border_fips = [
    '26005', '26019', '26021', '26023', '26027', '26041', '26043', '26053',
    '26059', '26071', '26083', '26089', '26091', '26101', '26105', '26109',
    '26115', '26121', '26127', '26131', '26139', '26149', '26159',
]

df_nb = df[~df['fips'].isin(border_fips)].copy()
df_nb_wage = df_nb[df_nb['neighbor_avg_rn_wage'].notna()].copy()
print(f"Wage gap sample: N={len(df_nb_wage)}, counties={df_nb_wage.fips.nunique()}, years={sorted(df_nb_wage.year.unique())}")

wage_controls_rn = baseline_controls + [
    'rn_per_10k_lag1',
    'ipeds_rn_per_10k_lag1',
    'rn_wage_gap'
]
twfe(df_nb_wage, 'rn_per_10k', wage_controls_rn, 'Wage competition - RN supply (non-border sample)')
twfe(df_nb_wage, 'rn_postings_per_10k', wage_controls_rn, 'Wage competition - RN postings (non-border sample)')

Wage gap sample: N=540, counties=60, years=[np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2021)]



=== Wage competition - RN supply (non-border sample) ===
  N=360  counties=60  years=6  within_R2=0.563  cov=clustered  FE=county+year



=== Wage competition - RN postings (non-border sample) ===
  N=418  counties=60  years=7  within_R2=0.061  cov=clustered  FE=county+year


(                           PanelOLS Estimation Summary                           
 Dep. Variable:     rn_postings_per_10k   R-squared:                        0.0611
 Estimator:                    PanelOLS   R-squared (Between):             -2.3500
 No. Observations:                  418   R-squared (Within):               0.1073
 Date:                 Wed, Jul 01 2026   R-squared (Overall):             -1.3378
 Time:                         13:37:00   Log-likelihood                   -1596.7
 Cov. Estimator:              Clustered                                           
                                          F-statistic:                      2.4803
 Entities:                           60   P-value                           0.0094
 Avg Obs:                        6.9667   Distribution:                   F(9,343)
 Min Obs:                        6.0000                                           
 Max Obs:                        7.0000   F-statistic (robust):          1.239e+04
    

**Section 5b notes.**
Twenty-three Michigan counties sharing a land border with Ohio, Indiana, or Wisconsin are
excluded from wage competition specifications. For these counties, neighbor wage averages
are computed from an incomplete set of adjacent labor markets — out-of-state neighbors are
unavailable in the ACS PUMS panel — causing the gap variable to be systematically
upward-biased.

`rn_wage_gap` and `lpn_wage_gap` inherit the endogeneity of own-county wages: a county's
wage is set in part by its nurse supply conditions, so the wage gap is not an exogenous
treatment. Do not interpret the coefficient as the causal effect of raising the county wage
gap. These results are included as focused tests of a spatial wage-competition mechanism,
with results reported for the restricted 60-county, wage-observed subsample only.

`lpn_wage_gap` is not estimated. Its unconditional correlation with LPN supply is r = -0.003,
indicating no systematic relationship, and it would only introduce sample loss.

Results from the full 83-county sample (excluding wage gap variables) are unaffected by
the border county restriction and remain the preferred main specifications.

---
## Appendix Models

### A1 — Age-decomposition restriction test

The baseline model includes `share_75plus` (old-old, 75+) and `share_65_74` (young-old, 65–74) separately. This appendix estimates the restricted model that collapses both into a single `share_65plus` coefficient — equivalent to constraining the two cohort effects to be equal. If the pipeline coefficient is stable across the restricted and unrestricted specs, the decomposition is informative about the aging mechanism but does not drive the headline. If it changes materially, the decomposition is load-bearing.

In [22]:
# A1: Restricted spec — collapse share_75plus + share_65_74 into single share_65plus
# Baseline drops both age-cohort vars and replaces with share_65plus
baseline_restricted = [c for c in baseline_controls
                        if c not in ('share_75plus', 'share_65_74')] + ['share_65plus']

print('Restricted spec controls:', baseline_restricted)
print()

twfe(df, 'rn_per_10k',  baseline_restricted + ['ipeds_rn_per_10k_lag1'],
     'A1 restricted - RN [share_65plus]', key_vars=['ipeds_rn_per_10k_lag1'])
twfe(df, 'lpn_per_10k', baseline_restricted + ['ipeds_lpn_per_10k_lag1'],
     'A1 restricted - LPN [share_65plus]', key_vars=['ipeds_lpn_per_10k_lag1'])

print('\n--- Baseline (unrestricted) for comparison ---')
twfe(df, 'rn_per_10k',  baseline_controls + ['ipeds_rn_per_10k_lag1'],
     'A1 baseline - RN [decomposed]', key_vars=['ipeds_rn_per_10k_lag1'])
twfe(df, 'lpn_per_10k', baseline_controls + ['ipeds_lpn_per_10k_lag1'],
     'A1 baseline - LPN [decomposed]', key_vars=['ipeds_lpn_per_10k_lag1'])


Restricted spec controls: ['population_growth_rate', 'hosp_beds_per_1k', 'nh_beds_per_65plus_ahrf', 'poverty_rate', 'bachelors_plus_share', 'share_65plus']


=== A1 restricted - RN [share_65plus] ===
  N=913  counties=83  years=11  within_R2=0.048  cov=clustered  FE=county+year
    ipeds_rn_per_10k_lag1: coef=0.0518  se=0.1351  p=0.7014

=== A1 restricted - LPN [share_65plus] ===
  N=911  counties=83  years=11  within_R2=0.148  cov=clustered  FE=county+year
    ipeds_lpn_per_10k_lag1: coef=0.4431  se=0.0656  p=0.0000

--- Baseline (unrestricted) for comparison ---



=== A1 baseline - RN [decomposed] ===
  N=913  counties=83  years=11  within_R2=0.088  cov=clustered  FE=county+year
    ipeds_rn_per_10k_lag1: coef=0.0235  se=0.1190  p=0.8432

=== A1 baseline - LPN [decomposed] ===
  N=911  counties=83  years=11  within_R2=0.164  cov=clustered  FE=county+year
    ipeds_lpn_per_10k_lag1: coef=0.4160  se=0.0554  p=0.0000


(                          PanelOLS Estimation Summary                           
 Dep. Variable:            lpn_per_10k   R-squared:                        0.1635
 Estimator:                   PanelOLS   R-squared (Between):             -0.1520
 No. Observations:                 911   R-squared (Within):               0.2828
 Date:                Wed, Jul 01 2026   R-squared (Overall):             -0.1289
 Time:                        13:37:00   Log-likelihood                   -2796.2
 Cov. Estimator:             Clustered                                           
                                         F-statistic:                      19.794
 Entities:                          83   P-value                           0.0000
 Avg Obs:                       10.976   Distribution:                   F(8,810)
 Min Obs:                      10.0000                                           
 Max Obs:                       11.000   F-statistic (robust):             44.373
                

### A1b - Outcome rescaled per 65+ population

Reviewer-requested. Re-estimate the baseline models with supply expressed **per
10k of the 65+ population** (the care-demand denominator) instead of per total
population. The numerator is unchanged (the licensed nurse count); only the
denominator changes, so coefficients are not directly comparable in magnitude to
the per-total-population models - the **sign and relative pattern** are the
object of interest.

In [23]:
twfe(df, 'rn_per_10k_65',  baseline_controls + ['ipeds_rn_per_10k_lag1'],
     'Appendix A1b - Baseline RN [per 65+]', key_vars=['ipeds_rn_per_10k_lag1'])
twfe(df, 'lpn_per_10k_65', baseline_controls + ['ipeds_lpn_per_10k_lag1'],
     'Appendix A1b - Baseline LPN [per 65+]', key_vars=['ipeds_lpn_per_10k_lag1'])

print('\nMean supply: per total population vs per 65+ population')
print(f"  RN : {df['rn_per_10k'].mean():.1f} per 10k total  vs  {df['rn_per_10k_65'].mean():.1f} per 10k of 65+")
print(f"  LPN: {df['lpn_per_10k'].mean():.1f} per 10k total  vs  {df['lpn_per_10k_65'].mean():.1f} per 10k of 65+")



=== Appendix A1b - Baseline RN [per 65+] ===
  N=913  counties=83  years=11  within_R2=0.052  cov=clustered  FE=county+year
    ipeds_rn_per_10k_lag1: coef=-0.3783  se=0.9486  p=0.6901

=== Appendix A1b - Baseline LPN [per 65+] ===
  N=911  counties=83  years=11  within_R2=0.092  cov=clustered  FE=county+year
    ipeds_lpn_per_10k_lag1: coef=1.4942  se=0.5014  p=0.0030

Mean supply: per total population vs per 65+ population
  RN : 123.8 per 10k total  vs  651.5 per 10k of 65+
  LPN: 30.7 per 10k total  vs  157.2 per 10k of 65+


**Appendix A1 notes — Age-decomposition restriction test.**

---

**Restriction test passes cleanly.** Collapsing `share_75plus` + `share_65_74` into a single `share_65plus` coefficient barely moves the pipeline estimates:

| Spec | RN pipeline (p) | LPN pipeline (p) |
|------|----------------|------------------|
| Restricted — `share_65plus` | 0.052 (0.701) | 0.443 (<0.001) |
| Unrestricted — decomposed | 0.024 (0.843) | 0.416 (<0.001) |

The LPN pipeline coefficient shifts by only 0.027 (6%) and remains highly significant. The RN null holds in both. The decomposition constraint is not binding for the headline result.

**R² cost of constraint.** LPN within-R² drops from 0.164 (unrestricted) to 0.148 (restricted) — 1.6pp lost by forcing equal cohort effects. This confirms the decomposition adds genuine explanatory power for LPN supply dynamics, even though it does not change the pipeline inference.

**The `share_65_74` story.** In the unrestricted baseline, `share_65_74`=−1.88 for LPN (p=0.052). Collapsing with `share_75plus`=+1.02 into a single `share_65plus` masks this opposing pattern — the restricted coefficient averages the two and produces a near-zero result. The decomposition reveals a genuine heterogeneity in how different elderly cohorts relate to LPN supply.

**Paper guidance.** Cite as: *'The pipeline result is robust to collapsing the age decomposition into a single 65+ share (LPN: 0.443 vs 0.416, Appendix A1). The decomposed baseline is preferred as it captures opposing cohort effects that the restricted specification obscures.'*

### A2 — Bartik/2SLS wage instrument (documents failure)

The shift-share Bartik instrument for local nursing wages is retained for transparency. We
report **first-stage** regressions of PUMS nursing wages on the Bartik instrument plus controls;
the diagnostic is the first-stage coefficient/F on the instrument, not a causal wage effect.
`bartik_iv` is the RN-relevant shift-share instrument; `bartik_iv_lpn` is the LPN-specific
construction (no separate `bartik_iv_rn` column exists in the data).

In [24]:
bartik_controls = baseline_controls

bartik_rn  = bartik_controls + ['bartik_iv']
bartik_lpn = bartik_controls + ['bartik_iv_lpn']

res_fs_rn,  tbl_fs_rn  = twfe(df, 'rn_pums_wage',  bartik_rn,  'Appendix - RN wage Bartik first stage (diagnostic)')
res_fs_lpn, tbl_fs_lpn = twfe(df, 'lpn_pums_wage', bartik_lpn, 'Appendix - LPN wage Bartik first stage (diagnostic)')

for res, var, lbl in [(res_fs_rn, 'bartik_iv', 'RN'), (res_fs_lpn, 'bartik_iv_lpn', 'LPN')]:
    if res is not None:
        f_inst = res.tstats[var] ** 2
        print(f"{lbl} first-stage instrument t={res.tstats[var]:.3f}  (t^2 ≈ single-instrument F = {f_inst:.3f}, "
              f"far below the Stock-Yogo weak-instrument threshold of 10)")


=== Appendix - RN wage Bartik first stage (diagnostic) ===
  N=747  counties=83  years=9  within_R2=-0.007  cov=clustered  FE=county+year



=== Appendix - LPN wage Bartik first stage (diagnostic) ===
  N=730  counties=83  years=9  within_R2=0.017  cov=clustered  FE=county+year
RN first-stage instrument t=1.990  (t^2 ≈ single-instrument F = 3.958, far below the Stock-Yogo weak-instrument threshold of 10)
LPN first-stage instrument t=-1.147  (t^2 ≈ single-instrument F = 1.316, far below the Stock-Yogo weak-instrument threshold of 10)


**Appendix A1b notes — Baseline models rescaled per 65+ population.**

---

**Scale.** Michigan counties average 651.5 RNs and 157.2 LPNs per 10,000 residents aged 65+, vs 123.8 and 30.7 per 10,000 total — roughly a 5× scaling factor. Coefficients not directly comparable in magnitude to main tables.

**RN null confirmed.** `ipeds_rn_per_10k_lag1`: coef=−0.378, se=0.949, p=0.690. Near-zero and highly insignificant — consistent with Section 1 null.

**LPN effect survives.** `ipeds_lpn_per_10k_lag1`: coef=1.494, se=0.501, p=0.003. Positive and significant. Pure denominator scaling predicts 0.416×(157.2/30.7)≈2.13; the actual 1.494 is attenuated (noisier 65+ denominator) but direction and significance intact.

**Model fit.** Within-R²=0.052 (RN), 0.092 (LPN) — slightly improved from the prior spec (0.020/0.086) due to the age decomposition absorbing more demographic variation, but still lower than per-total equivalents (0.088/0.164) due to intercensal noise in 65+ population estimates.

**Paper guidance.** Both headline findings replicate: *'Results are robust to expressing supply relative to the 65+ population (Appendix A1b), the primary care-demand denominator for nursing services.'*

**Appendix A2 notes.**
The shift-share (Bartik) instrument constructs local nursing wage variation from the
interaction of each county's 2014 industry employment shares with national industry-level
wage growth. Identification requires both relevance (initial industry mix transmits national
wage shocks to local nursing wages) and exclusion (industry shares affect nurse supply only
through wages).

In the Michigan-only sample, industry composition is too homogeneous to generate meaningful
cross-county variation in the instrument. The single-instrument first-stage F-statistics
(squared first-stage t-statistics) are below 1 in both the RN and LPN specifications — far
below the Stock-Yogo threshold of 10. The wage supply elasticity is therefore not identified
in this design. These first-stage diagnostics are reported for transparency only and should
not be cited as evidence on wage effects, and no second-stage 2SLS estimates are reported. A
multi-state extension would provide more identifying variation.

---
## Section 6 — Export Results

In [25]:
results_df = pd.concat([r for r in all_results if r is not None], ignore_index=True)
results_df = results_df.round(4)
results_df.to_csv('../outputs/mechanism_model_results.csv', index=False)

# Sample size summary
sample_summary = (
    results_df.drop_duplicates(subset=['label','outcome','n','counties','years','within_r2'])
    [['label','outcome','n','counties','years','within_r2']]
    .reset_index(drop=True)
)
sample_summary.to_csv('../outputs/mechanism_model_sample_sizes.csv', index=False)
print(sample_summary.to_string())
# ── Multiple testing note ────────────────────────────────────────────────────
# Across all sections, urbanicity strata, and robustness specs, 40+ significance
# tests are reported in this notebook. The headline LPN training result is the
# single pre-specified focal test; all other results are exploratory. The paper
# should include a one-line acknowledgment of this: 'The LPN training coefficient
# is the pre-specified focal test; all remaining results are exploratory and
# should be interpreted accordingly.' Do not apply ad hoc Bonferroni correction
# to the headline result — it was specified before estimation.
print('\nMultiple testing reminder: LPN training is the pre-specified focal test.')
print('All other reported results are exploratory.')


                                                                                   label               outcome    n  counties  years  within_r2
0                                                                     Baseline RN supply            rn_per_10k  913        83     11      0.088
1                                                                    Baseline LPN supply           lpn_per_10k  911        83     11      0.164
2                                                      No-FE compare [rn_per_10k] - TWFE            rn_per_10k  913        83     11      0.088
3                                                No-FE compare [rn_per_10k] - pooled OLS            rn_per_10k  913        83     11      0.520
4                                                     No-FE compare [lpn_per_10k] - TWFE           lpn_per_10k  911        83     11      0.164
5                                               No-FE compare [lpn_per_10k] - pooled OLS           lpn_per_10k  911        83     11    

### Cross-section comparability check

In [26]:
common_vars = baseline_controls + [
    'rn_per_10k_lag1', 'ipeds_rn_per_10k_lag1',
    'rn_age_55plus_pct_lag1', 'rn_age_under35_pct_lag1',
    'mean_commute_minutes', 'operating_margin_wmean', 'single_hospital_county'
]
df_common = df.dropna(subset=common_vars + ['rn_per_10k'])
print(f"Common sample: N={len(df_common)}, counties={df_common.fips.nunique()}, years={sorted(df_common.year.unique())}")
print("\nWhen comparing coefficients across Sections 1-5, re-estimate on this intersection sample —")
print("differences across sections may reflect sample composition rather than specification.")

Common sample: N=479, counties=73, years=[np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2020), np.int64(2021), np.int64(2023)]

When comparing coefficients across Sections 1-5, re-estimate on this intersection sample —
differences across sections may reflect sample composition rather than specification.


---
## Section 6b — County Targeting Shortlists

Descriptive priority flags from z-score composites on the most recent data year — **not** causal rankings. `rn_density_gap`/`lpn_density_gap` (r = 0.86 / 0.74 with own-supply, mechanically: gap = own supply − neighbor average) are used here only, never in TWFE.

In [27]:
TARGET_YEAR = int(df['year'].max())
snap = df[df['year'] == TARGET_YEAR].copy()
print(f"Target year for county shortlists: {TARGET_YEAR}  (N counties = {snap.fips.nunique()})")

def _z(s):
    s = pd.to_numeric(s, errors='coerce')
    sd = s.std(ddof=0)
    return (s - s.mean()) / sd if sd and not np.isnan(sd) else s * 0.0

snap['score_vacancy_rn']    =  _z(snap['rn_postings_per_10k'])   - _z(snap['rn_per_10k_lag1'])
snap['score_vacancy_lpn']   =  _z(snap['lpn_postings_per_10k'])  - _z(snap['lpn_per_10k_lag1'])
snap['score_retention_rn']  =  _z(snap['rn_age_55plus_pct'])   - _z(snap['rn_age_under35_pct'])  + _z(snap['mean_commute_minutes'])
snap['score_retention_lpn'] =  _z(snap['lpn_age_55plus_pct'])  - _z(snap['lpn_age_under35_pct']) + _z(snap['mean_commute_minutes'])
snap['score_training_lpn']  = -_z(snap['lpn_per_10k'])          - _z(snap['ipeds_lpn_per_10k_lag1'])
snap['score_employer']      = -_z(snap['operating_margin_wmean'])- _z(snap['rn_per_10k_lag1'])
# Density gap used descriptively here only
snap['score_spatial_rn']    = -_z(snap['rn_density_gap'])

target_cols = [
    'fips', 'county_name', 'year',
    'score_vacancy_rn', 'score_vacancy_lpn',
    'score_retention_rn', 'score_retention_lpn',
    'score_training_lpn', 'score_employer', 'score_spatial_rn'
]
county_targets = snap[target_cols].sort_values('score_vacancy_rn', ascending=False).reset_index(drop=True)
county_targets.to_csv('../outputs/mechanism_county_targets.csv', index=False)
print(county_targets.head(10).to_string())

Target year for county shortlists: 2023  (N counties = 83)
    fips county_name  year  score_vacancy_rn  score_vacancy_lpn  score_retention_rn  score_retention_lpn  score_training_lpn  score_employer  score_spatial_rn
0  26095        Luce  2023             3.983              0.249              -1.827               -5.230              -0.127           0.873             0.588
1  26025     Calhoun  2023             2.343              0.945              -0.535               -1.973              -4.103           0.975             0.108
2  26121    Muskegon  2023             2.146              1.588              -1.291               -0.243              -0.756           1.369               NaN
3  26057     Gratiot  2023             1.887              2.066              -0.911                1.309               0.503           0.649             0.715
4  26027        Cass  2023             1.823              1.186               1.230               -0.407               1.580           2.662      

---
## Policy-Mechanism Mapping Table

In [28]:
policy_mapping = pd.DataFrame([
    {
        'mechanism': 'Baseline distribution',
        'empirical_model': 'RN/LPN supply TWFE, distributional controls',
        'main_variables': 'share_75plus, hosp_beds_per_1k, nh_beds_per_65plus_ahrf, bachelors_plus_share, population_growth_rate',
        'policy_lever': 'Identifies where supply is located; not a direct policy lever',
        'causal_status': 'Descriptive only',
        'recommended_claim_strength': 'Descriptive'
    },
    {
        'mechanism': 'Vacancy pressure',
        'empirical_model': 'RN/LPN postings per 10k TWFE',
        'main_variables': 'rn/lpn_per_10k_lag1, ipeds_rn/lpn_per_10k_lag1, hpsa_prim_care',
        'policy_lever': 'Targeted recruitment incentives, loan repayment, shortage-county prioritization',
        'causal_status': 'Operational shortage-pressure; postings endogenous to supply',
        'recommended_claim_strength': 'Moderate — targeting, not causal effect'
    },
    {
        'mechanism': 'Training capacity',
        'empirical_model': 'RN/LPN supply TWFE with lagged IPEDS completions',
        'main_variables': 'ipeds_lpn_per_10k_lag1 (strong, but bootstrap-marginal: p≈0.12 vs asymptotic p≈0.003), ipeds_rn_per_10k_lag1 (null)',
        'policy_lever': 'LPN program expansion, clinical placements, preceptor incentives',
        'causal_status': 'Causal-leaning for LPNs (lag, common-denominator robust); wild-cluster bootstrap weakens significance to ~12% level; null for RNs',
        'recommended_claim_strength': 'Moderate-to-strong for LPNs (caveat bootstrap inference); null for RNs'
    },
    {
        'mechanism': 'Retention pressure',
        'empirical_model': 'Supply/postings TWFE with age composition and commute burden',
        'main_variables': 'rn/lpn_age_55plus_pct_lag1, rn/lpn_age_under35_pct_lag1, mean_commute_minutes',
        'policy_lever': 'Retention bonuses, scheduling flexibility, safety, mentorship, return-to-practice',
        'causal_status': 'Suggestive; age composition and commute are proxies for unobserved burnout',
        'recommended_claim_strength': 'Suggestive — data-limited'
    },
    {
        'mechanism': 'Employer financial capacity',
        'empirical_model': 'Supply TWFE with hospital operating margin',
        'main_variables': 'operating_margin_wmean, single_hospital_county',
        'policy_lever': 'Rural hospital stabilization, Medicaid reimbursement, nursing-home subsidies',
        'causal_status': 'Exploratory; margin endogenous to staffing; sign direction requires verification',
        'recommended_claim_strength': 'Exploratory only'
    },
    {
        'mechanism': 'Wage competition',
        'empirical_model': 'RN supply TWFE with neighbor wage gap (non-border subsample)',
        'main_variables': 'rn_wage_gap (non-border 60-county sample only)',
        'policy_lever': 'Competitive wage benchmarking, regional wage-equity policy',
        'causal_status': 'Descriptive; wage gap inherits endogeneity of own-county wages',
        'recommended_claim_strength': 'Descriptive — focused mechanism test'
    },
])
policy_mapping.to_csv('../outputs/mechanism_policy_mapping.csv', index=False)
policy_mapping[['mechanism', 'recommended_claim_strength']]

,mechanism,recommended_claim_strength
0,Baseline distribution,Descriptive
1,Vacancy pressure,"Moderate — targeting, not causal effect"
2,Training capacity,Moderate-to-strong for LPNs (caveat bootstrap ...
3,Retention pressure,Suggestive — data-limited
4,Employer financial capacity,Exploratory only
5,Wage competition,Descriptive — focused mechanism test


---
## Causal Interpretation Rules (Credibility Ladder)

Applied consistently in all interpretation cells above:

**Level 1 — Causal-leaning:** predetermined variables (lagged IPEDS completions, lagged supply). e.g. *"An increase in lagged LPN completions predicts higher subsequent LPN supply, consistent with a local training-to-workforce pipeline."* Caveat: residence/workplace mismatch, anticipatory expansion, common-denominator artifact, finite-cluster inference.

**Level 2 — Robust conditional association:** slow-moving, non-exogenous variables (`poverty_rate`, `bachelors_plus_share`, `share_75plus`). e.g. *"Counties with higher bachelor's degree attainment show higher nurse supply within-county over time."* Never: "Education causes nurse supply."

**Level 3 — Conditioning information only:** co-determined variables (`hosp_beds_per_1k`, `hpsa_prim_care`, `operating_margin_wmean`). e.g. *"Capacity variables are included as conditioning information. Their coefficients do not have clean causal interpretations."* Never call these policy levers directly. The **NP-substitution interaction** (`np_per_10k`) and the **RN part-time share** sit here too - descriptive conditioning information, not structural effects. The **no-FE / pooled** estimates are descriptive **between-county** associations, weaker than the within-county FE estimates, not stronger.

**Level 4 — Not identified:** 2SLS wage specification (Bartik instrument fails). e.g. *"The supply elasticity of wages is not identified in this design. A multi-state extension is required."*